In [3]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, sklearn, jupyterlab, ipywidgets, tqdm, pycountry
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas matplotlib plotly scikit-learn "jupyterlab>=3" "ipywidgets>=7.6" tqdm pycountry

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 10. EM-алгоритм та розділення суміші Гаусівських функцій

### **Це допоможе закріпити такі навички:**

- Попередньої підготовки даних до моделювання
- Роботу з бібліотеками аналізу даних

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити дані [World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness).***

Для виконання завдання необхідно виконати такі кроки:

1. **Інсталювати та імпортувати необхідні бібліотеки:** 
    - Необхідно буде інсталювати такі пакети:
	```bash
	!pip install plotly==5.20.0
	!pip install "jupyterlab>=3" "ipywidgets>=7.6"
	```

2. **Завантажити дані:**
    - З набору https://www.kaggle.com/datasets/unsdsn/world-happiness.
	```bash
	!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
	```

3. **Розпакувати дані:**
    ```bash
	!unzip WorldHappinessReport.zip
	```

4. **Прочитати дані та відобразити загальну інформацію про:**
	- Статистики
	- Типи ознак

5. **Побудувати діаграми розподілу числових ознак:**
    - Проаналізувати на відповідність чи не відповідність нормальному розподілу.

6. **Відібрати числових ознак та кореляційну матрицю:**
    - Виходячи із розуміння домену та даних відібрати певну кількість числових ознак
    - Відобразити кореляційну матрицю (*див. Тема 4. Вимірювання відстаней та подібностей в аналізі даних*)

7. **Зробити висновок про:**
    - Наявність та силу лінійного зв'язку між ознаками.

8. **Відобразити розподіл:**
    - Цільової ознаки (Happiness.Score або Happiness.Rank) за країнами.
    - Використовуючи наведений нижче код для побудови теплової мапи.
	```py
	fig = px.choropleth(data_dataframe,
						locations = "Country",
						color = "Happiness.Score",
						locationmode = "country names",
                    	)
	fig.update_layout(title = "Happiness Index 2017")
	fig.show()
	```

9. **Застосувати стандартизацію даних:**
    - Для приведення всіх значень до одного діапазону статистик.
    - Використовуючи функцію data_scale() та наступні перетворення
	```py
	def data_scale(data, scaler_type='minmax'):
	    from sklearn.preprocessing import MinMaxScaler
	    from sklearn.preprocessing import StandardScaler
	    from sklearn.preprocessing import Normalizer
	    if scaler_type == 'minmax':
	        scaler = MinMaxScaler()
	    if scaler_type == 'std':
	        scaler = StandardScaler()
	    if scaler_type == 'norm':
	        scaler = Normalizer()

	    scaler.fit(data)
	    res = scaler.transform(data)
	    return res

	data_scaled = data_scale(original_dataframe)
	df_scaled = pd.DataFrame(data_scaled, columns=[original_dataframe.columns])
	print(df_scaled.head())
	```

10. **Відобразити статистики:**
    - Отриманого стандартизованого набору даних та порівняти зі статистиками оригінального набору даних.
    - Зробити висновки.

11. **Побудувати модель кластеризації:**
	- Засобами функції `GaussianMixture()` бібліотеки `sklearn`.

12. **Побудувати теплову мапу:**
    - Для відображення розподілу країн за кластерами.

13. **Дослідити вплив:**
    - Різного набору ознак
    - Результат кластеризації

14. **Висновок:**
    - Зробити загальний висновок про відповідність результатів кластеризації оригінальному розподілу країн за ознакою.

**1. Імпорт необхідних бібліотек:**

In [4]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON (Мережа, Файлова система, Попередження)
import glob
import os
import shutil
from sklearn.exceptions import ConvergenceWarning
import textwrap
import time
import urllib.request
import zipfile
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import math
import numpy as np
import pandas as pd

# 3. МАШИННЕ НАВЧАННЯ (Кластеризація, Препроцесинг та Метрики)
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# 4. MLOps ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

# 5. ВІЗУАЛІЗАЦІЯ ТА UI (Plotly, IPywidgets & HTML)
import colorsys
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pycountry
from scipy.stats import multivariate_normal
import scipy.stats as stats

print("📦 Модулі архітектури імпортовано успішно!")

📦 Модулі архітектури імпортовано успішно!


**1.3. Конфігурація експерименту (Глобальні змінні):**

In [5]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА (MASTER SWITCH)
TARGET_YEAR = "2017"  # Доступні роки: "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"

# Офіційні набори даних (2015-2019) лежать в одному архіві. Сучасні (2020+) публікуються окремо.
KAGGLE_SOURCES = {
    "2015": "unsdsn/world-happiness",
    "2016": "unsdsn/world-happiness",
    "2017": "unsdsn/world-happiness",
    "2018": "unsdsn/world-happiness",
    "2019": "unsdsn/world-happiness",
    "2020": "mathurinache/world-happiness-report",
    "2021": "ajaypalsinghlo/world-happiness-report-2021",
    "2022": "mathurinache/world-happiness-report-2022",
    "2023": "ajaypalsinghlo/world-happiness-report-2023",
    "2024": "ajaypalsinghlo/world-happiness-report-2024"
}

if TARGET_YEAR not in KAGGLE_SOURCES:
    raise SystemExit(f"❌ Критична Помилка Конфігурації: Року '{TARGET_YEAR}' не існує в системі!\n"
                     f"   👉 Доступні роки: {list(KAGGLE_SOURCES.keys())}")

DATA_DIR     = "WorldHappinessDataSet"       # Папка для ізольованого збереження всіх сирих даних
DATASET_URL  = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_SOURCES[TARGET_YEAR]}"
ZIP_PATH     = os.path.join(DATA_DIR, f"dataset_{TARGET_YEAR}.zip")
CSV_FILENAME = os.path.join(DATA_DIR, f"{TARGET_YEAR}.csv") # Динамічний шлях до потрібного файлу
EXPORT_CSV_PATH = os.path.join(DATA_DIR, f"{TARGET_YEAR}_clustered_result.csv") # Шлях для збереження фінальної таблиці з мітками ШІ

# 2. СТРУКТУРА ДАНИХ ТА СЕМАНТИЧНИЙ ШАР (Вирішення проблеми Schema Drift)
# Абстрактні шаблони (Епохи Kaggle)
_SCHEMAS = {
    # --- СТАРА ЕРА (2015-2017) ---
    "v1_classic": { # 2015, 2016
        "country": "Country", "target": "Happiness Score",
        "cols": {"gdp": "Economy (GDP per Capita)", "social": "Family", "health": "Health (Life Expectancy)", "freedom": "Freedom", "generosity": "Generosity", "corruption": "Trust (Government Corruption)"}
    },
    "v1_dotted": {  # 2017
        "country": "Country", "target": "Happiness.Score",
        "cols": {"gdp": "Economy..GDP.per.Capita.", "social": "Family", "health": "Health..Life.Expectancy.", "freedom": "Freedom", "generosity": "Generosity", "corruption": "Trust..Government.Corruption."}
    },

	# --- ПЕРЕХІДНА ЕРА (2018-2019) ---
    "v2_trans": {   # 2018, 2019
        "country": "Country or region", "target": "Score",
        "cols": {"gdp": "GDP per capita", "social": "Social support", "health": "Healthy life expectancy", "freedom": "Freedom to make life choices", "generosity": "Generosity", "corruption": "Perceptions of corruption"}
    },

    # --- СУЧАСНА ЕРА (2020-2024) ---
    "v3_modern": {  # 2020, 2021, 2023
        "country": "Country name", "target": "Ladder score",
        "cols": {"gdp": "Logged GDP per capita", "social": "Social support", "health": "Healthy life expectancy", "freedom": "Freedom to make life choices", "generosity": "Generosity", "corruption": "Perceptions of corruption"}
    },
    "v3_2022": {    # 2022
        "country": "Country", "target": "Happiness score",
        "cols": {"gdp": "Explained by: GDP per capita", "social": "Explained by: Social support", "health": "Explained by: Healthy life expectancy", "freedom": "Explained by: Freedom to make life choices", "generosity": "Explained by: Generosity", "corruption": "Explained by: Perceptions of corruption"}
    },
    "v4_2024": {    # 2024 (Новий сюрприз від Kaggle: колонки Explained by + Country name)
        "country": "Country name", "target": "Ladder score",
        "cols": {"gdp": "Explained by: Log GDP per capita", "social": "Explained by: Social support", "health": "Explained by: Healthy life expectancy", "freedom": "Explained by: Freedom to make life choices", "generosity": "Explained by: Generosity", "corruption": "Explained by: Perceptions of corruption"}
    }
}

YEAR_TO_SCHEMA = {
    "2015": "v1_classic",
    "2016": "v1_classic",
    "2017": "v1_dotted",
    "2018": "v2_trans",
    "2019": "v2_trans",
    "2020": "v3_modern",
    "2021": "v3_modern",
    "2022": "v3_2022",
    "2023": "v3_modern",
    "2024": "v4_2024"
}

# ПАНЕЛЬ КЕРУВАННЯ ВИМІРАМИ (Гнучкий вибір ознак для GMM)
ACTIVE_DIMENSIONS = ["gdp", "social", "health", "freedom", "generosity", "corruption"]  # Виміри для тестування (можна вибрати будь-яку комбінацію, наприклад: ["gdp", "health"] для експерименту з 2-ма ознаками)

CURRENT_SCHEMA   = _SCHEMAS[YEAR_TO_SCHEMA[TARGET_YEAR]]
COUNTRY_COL      = CURRENT_SCHEMA["country"]             							  # Динамічна колонка країни (змінювалась у 2018 та 2020)
TARGET_METRIC    = CURRENT_SCHEMA["target"]              							  # Головна цільова метрика (Індекс щастя)
FEATURES_FULL    = [CURRENT_SCHEMA["cols"][dim] for dim in ACTIVE_DIMENSIONS] 		  # Повний набір соціально-економічних ознак для GMM
FEATURES_MINI    = [CURRENT_SCHEMA["cols"]["gdp"], CURRENT_SCHEMA["cols"]["health"]]  # Зменшений набір (ВВП та Здоров'я) для дослідження розмірності


# 3. МАШИННЕ НАВЧАННЯ (GMM) ТА СЕРІАЛІЗАЦІЯ
N_CLUSTERS       = 3          # Кількість кластерів: задає число прихованих Гаусівських розподілів (Високий, Середній, Низький рівень)
COVARIANCE_TYPE  = 'full'     # Форма матриці коваріації (геометрія кластерів): 'full' - різні еліпси під будь-яким кутом | 'tied' - однакова форма та нахил для всіх | 'diag' - еліпси строго паралельні осям координат | 'spherical' - ідеальні круглі сфери різного радіусу
GMM_INIT_PARAMS  = 'kmeans'   # Стратегія стартової ініціалізації (Крок 0 для EM): 'kmeans' - розумний розвідник для надійного старту | 'random' - повністю випадкові координати в просторі | 'random_from_data' - випадкові реальні точки з набору даних
SCALER_TYPE      = 'std'      # Алгоритм масштабування простору ознак: 'std' - центрує дисперсію навколо нуля (ідеально для GMM) | 'minmax' - жорстко стискає дані в межі від 0 до 1 | 'norm' - нормує самі вектори по їхній абсолютній довжині
N_INIT           = 15         # Кількість перезапусків EM-алгоритму: захист від застрягання моделі в поганих локальних мінімумах
RANDOM_STATE     = 42         # Фіксація генератора псевдовипадкових чисел: гарантує 100% відтворюваність результатів експерименту

MODEL_DIR        = "GMM_Models"    # Папка для збереження серіалізованих об'єктів
MODEL_PATH       = os.path.join(MODEL_DIR, f"gmm_{TARGET_YEAR}_{COVARIANCE_TYPE}_{GMM_INIT_PARAMS}_model.pkl") # Динамічне ім'я моделі
SCALER_PATH      = os.path.join(MODEL_DIR, f"scaler_{TARGET_YEAR}_{SCALER_TYPE}.pkl")                          # Динамічне ім'я скейлера


# 4. ВІЗУАЛІЗАЦІЯ, UI ТА ГЕО-МАПІНГ
PLOT_TEMPLATE           = "plotly_dark"                         # Темна тема для інтерактивних графіків Plotly
MAP_LOCATION_MODE       = "country names"                       # Режим розпізнавання країн для мап Choropleth
COLOR_SCALE_HAPPINESS   = "Viridis"                             # Безперервний градієнт для оригінального індексу щастя
COLOR_PALETTE_MINI      = px.colors.qualitative.Pastel          # Пастельні кольори для експерименту зі зменшеною розмірністю
CLUSTER_LABELS          = {0: "Низький рівень", 1: "Середній рівень", 2: "Високий рівень"}         # Бізнес-назви наших 3-х кластерів
TABLE_PROPS             = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'center'}
DESCRIBE_CMAP           = 'YlGn'                                # Кольорова схема (Yellow-Green) для підсвічування описових статистик
BASE_COLORS             = ['#ff4d4d', '#ffcc00', '#00cc66']     # Базові семантичні кольори (Червоний, Жовтий, Зелений)

# 🎨 Динамічний генератор кольорів (на випадок якщо N_CLUSTERS > 3)
class ColorGenerator:
    def __init__(self, palette):
        self.palette = palette
        self.cols = len(palette)

    def _convert(self, hex_color, to_hsv=True):
        hex_c = hex_color.lstrip('#')
        rgb = tuple(int(hex_c[i:i+2], 16)/255.0 for i in (0, 2, 4))
        return colorsys.rgb_to_hsv(*rgb) if to_hsv else rgb

    def __call__(self, index):
        if index < len(self.palette): 
            return self.palette[index]

        col, row = index % self.cols, index // self.cols
        h, s, v = self._convert(self.palette[col])

        nh = (h + (row * 0.08)) % 1.0
        ns = max(0.4, s - (row % 3) * 0.15)
        nv = max(0.4, 1.0 - (row % 4) * 0.15)

        rgb = colorsys.hsv_to_rgb(nh, ns, nv)
        return f"#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}"

_color_gen = ColorGenerator(BASE_COLORS)
# Генеруємо палітру точно під обрану кількість кластерів (або мінімум 3 для сумісності)
COLOR_PALETTE_FULL = [_color_gen(i) for i in range(max(N_CLUSTERS, 3))]

# 🏷️ Динамічна генерація міток (запобіжник від KeyError)
_base_labels = {0: "Низький рівень", 1: "Середній рівень", 2: "Високий рівень"}
CLUSTER_LABELS = {i: _base_labels.get(i, f"Рівень {i+1}") for i in range(max(N_CLUSTERS, 3))}

FEATURE_TRANSLATIONS = {    # Переклад соціально-економічних ознак
    # Ідентифікатори
    "Country": "Країна", "Country name": "Країна", "Country or region": "Країна або регіон", "Region": "Регіон", "Regional indicator": "Регіональний індикатор",
    # Цільові метрики
    "Happiness Score": "Індекс щастя", "Happiness.Score": "Індекс щастя", "Score": "Індекс щастя", "Ladder score": "Індекс щастя", "Happiness score": "Індекс щастя",
    "Happiness Rank": "Рейтинг щастя", "Happiness.Rank": "Рейтинг щастя", "Overall rank": "Загальний рейтинг", "RANK": "Рейтинг щастя",
    # Похибки
    "Standard Error": "Стандартна похибка", "Standard error of ladder score": "Стандартна похибка індексу", "Lower Confidence Interval": "Нижній довірчий інтервал", "Upper Confidence Interval": "Верхній довірчий інтервал", "Whisker.high": "Верхня межа (Whisker)", "Whisker.low": "Нижня межа (Whisker)", "upperwhisker": "Верхня межа (Whisker)", "lowerwhisker": "Нижня межа (Whisker)", "Whisker-high": "Верхня межа (Whisker)", "Whisker-low": "Нижня межа (Whisker)",
    # Економіка
    "Economy (GDP per Capita)": "ВВП на душу населення", "Economy..GDP.per.Capita.": "ВВП на душу населення", "GDP per capita": "ВВП на душу населення", "Logged GDP per capita": "ВВП на душу населення", "Explained by: Log GDP per capita": "ВВП на душу населення", "Explained by: GDP per capita": "ВВП на душу населення",
    # Соціум
    "Family": "Соціальна підтримка", "Social support": "Соціальна підтримка", "Explained by: Social support": "Соціальна підтримка",
    # Здоров'я
    "Health (Life Expectancy)": "Тривалість здорового життя", "Health..Life.Expectancy.": "Тривалість здорового життя", "Healthy life expectancy": "Тривалість здорового життя", "Explained by: Healthy life expectancy": "Тривалість здорового життя",
    # Свобода
    "Freedom": "Свобода вибору", "Freedom to make life choices": "Свобода вибору", "Explained by: Freedom to make life choices": "Свобода вибору",
    # Корупція
    "Trust (Government Corruption)": "Сприйняття корупції", "Trust..Government.Corruption.": "Сприйняття корупції", "Perceptions of corruption": "Сприйняття корупції", "Explained by: Perceptions of corruption": "Сприйняття корупції",
    # Щедрість
    "Generosity": "Щедрість", "Explained by: Generosity": "Щедрість",
    # Дистопія
    "Dystopia Residual": "Дистопія + залишок", "Dystopia.Residual": "Дистопія + залишок", "Ladder score in Dystopia": "Базовий індекс Дистопії", "Dystopia + residual": "Дистопія + залишок", "Dystopia (1.83) + residual": "Дистопія + залишок"
}

MANUAL_ISO_MAPPING = {
    # Європа
    "Czechia": "CZE", "Czech Republic": "CZE", "Macedonia": "MKD", "North Macedonia": "MKD", 
    "Kosovo": "XKX", "Kosovo*": "XKX", "Russia": "RUS", "Russian Federation": "RUS", "Moldova": "MDA", "Republic of Moldova": "MDA",
    # Кіпр
    "North Cyprus": "CYP", "Northern Cyprus": "CYP", "North Cyprus*": "CYP", "Cyprus": "CYP",
    # Туреччина
    "Turkiye": "TUR", "Turkey": "TUR", "Türkiye": "TUR",
    # Китай та Азія
    "Taiwan": "TWN", "Taiwan Province of China": "TWN", "Hong Kong": "HKG", "Hong Kong S.A.R.": "HKG", 
    "Hong Kong S.A.R. of China": "HKG", "Hong Kong S.A.R., China": "HKG", "Macao S.A.R. of China": "MAC",
    "Syria": "SYR", "Syrian Arab Republic": "SYR", "Iran": "IRN", "Iran, Islamic Republic of": "IRN", 
    "South Korea": "KOR", "Korea, South": "KOR", "Laos": "LAO", "Lao People's Democratic Republic": "LAO", 
    "Vietnam": "VNM", "Viet Nam": "VNM", "United Arab Emirates": "ARE",
    # Палестина
    "State of Palestine": "PSE", "Palestinian Territories": "PSE", "Palestinian Territories*": "PSE",
    # Африка
    "Congo (Brazzaville)": "COG", "Republic of the Congo": "COG", "Congo (Kinshasa)": "COD", "Democratic Republic of the Congo": "COD", 
    "Ivory Coast": "CIV", "Cote d'Ivoire": "CIV", "Côte d'Ivoire": "CIV", "Swaziland": "SWZ", "Swaziland*": "SWZ", "Eswatini": "SWZ", "Eswatini, Kingdom of": "SWZ",
    "Gambia": "GMB", "The Gambia": "GMB", "Gambia*": "GMB", "Somaliland region": "SOM", "Somaliland Region": "SOM", "Somalia": "SOM",
    # Америка
    "Trinidad and Tobago": "TTO", "Trinidad & Tobago": "TTO", "United States": "USA", "United States of America": "USA", 
    "Venezuela": "VEN", "Venezuela, Bolivarian Republic of": "VEN", "Bolivia": "BOL", "Bolivia, Plurinational State of": "BOL"
}

# --- 5. ТИХИЙ ПІДРАХУНОК КРАЇН У НАБОРІ ДАНИХ ---
try:
    _temp_df = pd.read_csv(CSV_FILENAME)
    _num_countries = len(_temp_df)
except Exception:
    _num_countries = "Файл ще не завантажено"

print("⚙️ Глобальні константи ініціалізовано!")
print(f"   📰 Рік: {TARGET_YEAR}")
print(f"   🌍 Кількість країн у наборі: {_num_countries}")
print(f"   🔗 Джерело: {KAGGLE_SOURCES[TARGET_YEAR]}")
print(f"   📊 Ознаки: {len(FEATURES_FULL)} вимірів {ACTIVE_DIMENSIONS}")
print(f"   🤖 Машинне навчання: GMM({COVARIANCE_TYPE}, {GMM_INIT_PARAMS}) + {SCALER_TYPE} Scaler")

⚙️ Глобальні константи ініціалізовано!
   📰 Рік: 2017
   🌍 Кількість країн у наборі: 155
   🔗 Джерело: unsdsn/world-happiness
   📊 Ознаки: 6 вимірів ['gdp', 'social', 'health', 'freedom', 'generosity', 'corruption']
   🤖 Машинне навчання: GMM(full, kmeans) + std Scaler


**1.7. Приклад на HTML (Анатомія GMM):**

In [6]:
C_RAW = "#888888"

cluster_spans = " | ".join([
    f'<span style="color:{COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]}">{CLUSTER_LABELS.get(i, f"Кластер {i+1}")}</span>' 
    for i in range(N_CLUSTERS)
])

cov_desc = {
    'full': 'змінюють форму еліпсів під будь-яким кутом',
    'diag': 'розтягують еліпси строго паралельно осям',
    'spherical': 'розширюють або звужують ідеальні сфери',
    'tied': 'змінюють розмір (але форма лишається однаковою для всіх кластерів)'
}.get(COVARIANCE_TYPE, 'змінюють форму еліпсів')

html_em_pipeline = f"""
<div style="font-family: sans-serif; max-width: 900px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333; margin: auto;">
    <h2 style="color: #00c3ff; text-align: center; margin-top: 0;">🧠 Анатомія GMM: Що робить EM-алгоритм з країнами?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {C_RAW}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий простір (Дані після {SCALER_TYPE} Scaler)</div>
        <div style="color: {C_RAW}; font-size: 15px; margin-top: 5px; font-style: italic;">
            Маємо N країн у багатовимірному просторі ознак (ВВП, Здоров'я, Свобода...).<br>
            Усі точки "сірі", алгоритм ще нічого не знає про кластери.
        </div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Ініціалізація (Метод '{GMM_INIT_PARAMS}')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px;">
            ШІ генерує {N_CLUSTERS} стартових центрів мас (Гаусівських розподілів).<br>
            Кожен має свій центр <b>(&mu;)</b> та матрицю коваріації <b>(&Sigma;)</b>.
        </div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇ ♻️ Цикл EM-алгоритму ♻️ ⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: E-крок (Expectation / Очікування)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px;">
            Обчислення м'якої ймовірності (Soft Clustering) за формулою Баєса:<br>
            <i>"Країна Х належить до Кластера-1 на 10%, Кластера-2 на 85%, Кластера-3 на 5%".</i>
        </div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: M-крок (Maximization / Максимізація)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px;">
            Оновлення параметрів дзвонів: <b>Нові &mu;</b> тягнуться до скупчень точок, <b>Нові &Sigma;</b> {cov_desc}.
        </div>
    </div>

    <div style="text-align: center; color: #00ffcc; font-size: 20px; margin-top: 10px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed #00ffcc; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Збіжність (Convergence)</div>
        <div style="color: #00ffcc; font-size: 18px; margin-top: 10px; font-family: monospace;">[ {cluster_spans} ]</div>
    </div>
</div>
"""

print("Красивий Вивід (Інтерактивна схема логіки алгоритму):")
display(HTML(html_em_pipeline))

Красивий Вивід (Інтерактивна схема логіки алгоритму):


**2. Завантажити дані:**

In [7]:
def is_valid_zip(filepath):
    if not os.path.exists(filepath) or not zipfile.is_zipfile(filepath):
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

def download_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"⏳ Завантаження архіву у папку '{DATA_DIR}'...")
    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
        print("✅ Завантаження завершено.")
    except Exception as e:
        print(f"❌ Мережева помилка завантаження: {e}")

if os.path.exists(ZIP_PATH):
    print("🔍 Перевірка цілісності існуючого архіву...")
    if not is_valid_zip(ZIP_PATH):
        print("🪫 Архів пошкоджено. Видаляємо та завантажуємо наново...")
        os.remove(ZIP_PATH)
        download_dataset()
    else:
        print("🔋 Архів цілий. Пропускаємо мережевий запит.")
else:
    download_dataset()

🔍 Перевірка цілісності існуючого архіву...
🔋 Архів цілий. Пропускаємо мережевий запит.


**3. Розпакувати дані:**

In [8]:
if os.path.exists(CSV_FILENAME):
    print(f"⚡ Файл '{CSV_FILENAME}' вже розпаковано та готовий до роботи.")
elif os.path.exists(ZIP_PATH):
    if is_valid_zip(ZIP_PATH):
        print(f"📦 Аналізуємо вміст архіву '{ZIP_PATH}'...")
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
                csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                if not csv_files:
                    raise Exception("В архіві немає CSV файлів!")

                expected_file_suffix = f"{TARGET_YEAR}.csv"
                target_csv_in_zip = next((f for f in csv_files if f.endswith(expected_file_suffix)), None)

                if not target_csv_in_zip:
                    print(f"   ⚠️ Доступні файли в архіві: {csv_files}")
                    raise Exception(f"Файл для {TARGET_YEAR} року не знайдено в архіві!")

                print(f"   🎯 Знайдено цільовий файл: '{target_csv_in_zip}'")

                tmp_csv_path = CSV_FILENAME + ".tmp"

                try:
                    print(f"   ⚙️ Витягуємо '{target_csv_in_zip}' атомарно...")
                    with zip_ref.open(target_csv_in_zip) as source, open(tmp_csv_path, "wb") as target:
                        shutil.copyfileobj(source, target)

                    os.replace(tmp_csv_path, CSV_FILENAME)
                    print(f"✅ Успіх! Файл '{CSV_FILENAME}' (дані {TARGET_YEAR} року) збережено безпечно.")

                except PermissionError:
                    raise Exception(f"Файл {CSV_FILENAME} заблоковано іншою програмою. Закрийте Excel або інші скрипти.")
                except Exception as extract_err:
                    raise Exception(f"Помилка фізичного запису на диск: {extract_err}")
                finally:
                    if os.path.exists(tmp_csv_path):
                        os.remove(tmp_csv_path)

        except Exception as e:
            print(f"❌ Системна помилка під час роботи з архівом: {e}")
    else:
        print("❌ Критична помилка: Архів досі пошкоджений.")
else:
    print("❌ Помилка: Архів не знайдено. Перезапустіть попередній блок завантаження.")

⚡ Файл 'WorldHappinessDataSet/2017.csv' вже розпаковано та готовий до роботи.


**4. Прочитати дані та відобразити загальну інформацію:**

In [9]:
print(f"📂 Завантаження набору даних з файлу: {CSV_FILENAME}\n")
try:
    df = pd.read_csv(CSV_FILENAME)
    df[COUNTRY_COL] = df[COUNTRY_COL].str.replace('*', '', regex=False).str.strip()
except Exception as e:
    raise SystemExit(f"❌ Помилка читання CSV файлу: {e}")

required_cols = FEATURES_FULL + [COUNTRY_COL, TARGET_METRIC]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise SystemExit(f"❌ Schema Drift Error! У наборі даних за {TARGET_YEAR} рік відсутні очікувані колонки:\n{missing_cols}\nОновіть SCHEMA_MAPPING у Блоці 1.")
else:
    print(f"✅ Схема даних підтверджена: знайдено всі {len(required_cols)} критичних колонок.\n")

print("Красивий Вивід - Перші 5 рядків набору даних:")
display(df.head().style.background_gradient(cmap='Blues').set_properties(**TABLE_PROPS))

print("Технічний Вивід:\nОписові статистики:")
display(df.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("Інформація про типи ознак та пропуски:")
df.info()

📂 Завантаження набору даних з файлу: WorldHappinessDataSet/2017.csv

✅ Схема даних підтверджена: знайдено всі 8 критичних колонок.

Красивий Вивід - Перші 5 рядків набору даних:


,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual
0,Norway,1,7.537000,7.594445,7.479556,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027
1,Denmark,2,7.522000,7.581728,7.462272,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707
2,Iceland,3,7.504000,7.622030,7.385970,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715
3,Switzerland,4,7.494000,7.561772,7.426227,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716
4,Finland,5,7.469000,7.527542,7.410458,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182


Технічний Вивід:
Описові статистики:


,count,mean,std,min,25%,50%,75%,max
Happiness.Rank,155.0000,78.0000,44.8888,1.0000,39.5000,78.0000,116.5000,155.0000
Happiness.Score,155.0000,5.3540,1.1312,2.6930,4.5055,5.2790,6.1015,7.5370
Whisker.high,155.0000,5.4523,1.1185,2.8649,4.6082,5.3700,6.1946,7.6220
Whisker.low,155.0000,5.2557,1.1450,2.5211,4.3750,5.1932,6.0065,7.4796
Economy..GDP.per.Capita.,155.0000,0.9847,0.4208,0.0000,0.6634,1.0646,1.3180,1.8708
Family,155.0000,1.1889,0.2873,0.0000,1.0426,1.2539,1.4143,1.6106
Health..Life.Expectancy.,155.0000,0.5513,0.2371,0.0000,0.3699,0.6060,0.7230,0.9495
Freedom,155.0000,0.4088,0.1500,0.0000,0.3037,0.4375,0.5166,0.6582
Generosity,155.0000,0.2469,0.1348,0.0000,0.1541,0.2315,0.3238,0.8381
Trust..Government.Corruption.,155.0000,0.1231,0.1017,0.0000,0.0573,0.0898,0.1533,0.4643


Інформація про типи ознак та пропуски:
<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        155 non-null    str    
 1   Happiness.Rank                 155 non-null    int64  
 2   Happiness.Score                155 non-null    float64
 3   Whisker.high                   155 non-null    float64
 4   Whisker.low                    155 non-null    float64
 5   Economy..GDP.per.Capita.       155 non-null    float64
 6   Family                         155 non-null    float64
 7   Health..Life.Expectancy.       155 non-null    float64
 8   Freedom                        155 non-null    float64
 9   Generosity                     155 non-null    float64
 10  Trust..Government.Corruption.  155 non-null    float64
 11  Dystopia.Residual              155 non-null    float64
dtypes: float64(10), int64(

**5. Побудувати діаграми розподілу числових ознак:**

In [10]:
raw_features = [TARGET_METRIC] + FEATURES_FULL
features_to_plot = [f for f in raw_features if f in df.columns]

print(f"📈 Аналіз нормальності розподілу для {len(features_to_plot)} ознак...\n")

num_plots = len(features_to_plot)
max_logical_cols = 3
rows = math.ceil(num_plots / max_logical_cols) if num_plots > 0 else 1

clean_titles = []
for f in features_to_plot:
    base_name = FEATURE_TRANSLATIONS.get(f, str(f).replace('.', ' ').replace('_', ' ').strip())

    clean_series = df[f].astype(str).str.replace(',', '.', regex=False)
    data = pd.to_numeric(clean_series, errors='coerce').dropna()

    if len(data) > 1:
        mu, std = data.mean(), data.std()
        clean_titles.append(f"{base_name}<br>(μ = {mu:.2f}, σ = {std:.2f})")
    else:
        clean_titles.append(f"{base_name}")

grid_specs = []
for r in range(rows):
    plots_in_row = min(max_logical_cols, num_plots - r * max_logical_cols)
    if plots_in_row == 3:
        grid_specs.append([{'colspan': 2}, None, {'colspan': 2}, None, {'colspan': 2}, None])
    elif plots_in_row == 2:
        grid_specs.append([None, {'colspan': 2}, None, {'colspan': 2}, None, None])
    elif plots_in_row == 1:
        grid_specs.append([None, None, {'colspan': 2}, None, None, None])

fig_dist = make_subplots(
    rows=rows, cols=6,
    specs=grid_specs,
    subplot_titles=clean_titles,
    vertical_spacing=0.12,  
    horizontal_spacing=0.04 
)

for i, feature in enumerate(features_to_plot):
    r = (i // max_logical_cols) + 1
    plot_idx_in_row = i % max_logical_cols
    plots_in_this_row = min(max_logical_cols, num_plots - (r - 1) * max_logical_cols)

    if plots_in_this_row == 3:
        c = plot_idx_in_row * 2 + 1
    elif plots_in_this_row == 2:
        c = 2 + plot_idx_in_row * 2
    else:
        c = 3

    clean_series = df[feature].astype(str).str.replace(',', '.', regex=False)
    data = pd.to_numeric(clean_series, errors='coerce').dropna()

    fig_dist.add_trace(
        go.Histogram(
            x=data, histnorm='probability density', 
            name=f"{feature}", marker_color='#00c3ff', opacity=0.6, nbinsx=25,
            hovertemplate="<b>Діапазон значень:</b> %{x}<br><b>Емпірична щільність:</b> %{y:.4f}<extra></extra>"
        ), row=r, col=c
    )

    if len(data) > 1:
        mu, std = data.mean(), data.std()
        
        if std > 0:
            x_curve = np.linspace(data.min(), data.max(), 100)
            y_curve = stats.norm.pdf(x_curve, mu, std)

            fig_dist.add_trace(
                go.Scatter(
                    x=x_curve, y=y_curve, mode='lines', 
                    name=f"Ідеальний Гаусс", line=dict(color='#ffd700', width=3, dash='dot'),
                    hovertemplate="<b>Ідеальний Гаусс</b><br>Значення ознаки: %{x:.2f}<br>Теоретична щільність: %{y:.4f}<extra></extra>"
                ), row=r, col=c
            )

    fig_dist.update_xaxes(title_text="Значення", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=r, col=c)
    fig_dist.update_yaxes(title_text="Щільність", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=r, col=c)

safe_year = globals().get('TARGET_YEAR', 'Невідомий')
dynamic_legend_y = -0.15 / rows if rows > 0 else -0.15

fig_dist.update_layout(
    height=350 * rows + 120, 
    width=1500,
    title_text=f"📊 Перевірка на нормальність ({safe_year} рік): Реальний розподіл vs Ідеальний Дзвін Гаусса", 
    title_x=0.5,
    template=PLOT_TEMPLATE,
    showlegend=False,
    hovermode="x unified",
    margin=dict(b=120, t=120)
)

fig_dist.add_annotation(
    x=0.5, y=dynamic_legend_y, xref="paper", yref="paper",
    text="🟡 <b>Жовтий пунктир</b> — ідеальна математична модель (Дзвін Гаусса).<br>🟦 <b>Блакитні стовпці</b> — реальний емпіричний розподіл ознаки в наборі даних.",
    showarrow=False, font=dict(size=14, color="#cccccc"), align="center", xanchor="center", yanchor="top"
)

print("Красивий Вивід:")
fig_dist.show()

📈 Аналіз нормальності розподілу для 7 ознак...

Красивий Вивід:


**5.1. Висновок до Кроку 5:**

### Аналіз нормальності розподілу

Для кожної ознаки ми побудували ідеальну теоретичну криву (жовтий пунктир), яка описується рівнянням щільності одномірного нормального розподілу:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

де $\mu$ — математичне сподівання (середнє значення), а $\sigma$ — стандартне відхилення ознаки. Порівнюючи емпіричні гістограми з цією теоретичною моделлю, робимо такі висновки щодо природи соціально-економічних даних:

1. **Відсутність ідеальної нормальності:** Більшість ознак у наборі даних **не мають** ідеально симетричного нормального розподілу. Реальні макроекономічні дані схильні до перекосів (Skewness) та нетипових викидів.
2. **Лівостороння асиметрія (Negative Skew):** Ознаки `Economy (GDP) [ВВП на душу населення]` та `Health (Life Expectancy) [Тривалість здорового життя]` помітно зміщені вправо. Це означає, що у світі переважають країни із середнім та високим рівнем життя, тоді як країни з абсолютною бідністю формують довгий, але тонкий лівий "хвіст".
3. **Правостороння асиметрія (Positive Skew):** Ознака `Trust (Government Corruption) [Сприйняття корупції]` має яскраво виражений експоненційний спад. Переважна більшість країн має дуже низький індекс довіри до уряду, і лише одиничні геополітичні "аномалії" (як-от країни Скандинавії) мають високі показники, утворюючи правий "хвіст".
4. **Вплив на GMM-кластеризацію:** Жорсткий алгоритм K-Means працює погано з такими асиметричними даними, оскільки намагається вписати точки в ідеальні сфери. Натомість `GaussianMixture` (Модель суміші Гаусів) чудово впорається з цим завданням з двох причин:
    - Використання повної матриці коваріації (`covariance_type='full'`) дозволяє кластерам набувати форми витягнутих еліпсів.
    - Математично доведено, що лінійна комбінація кількох Гаусівських розподілів здатна апроксимувати будь-який, навіть найскладніший і найбільш асиметричний розподіл даних.

**6. Відібрати числових ознак та кореляційну матрицю:**

In [11]:
print("🔍 Відбір числових ознак для аналізу...")

selected_columns = [TARGET_METRIC] + [f for f in FEATURES_FULL if f in df.columns]
df_selected = df[selected_columns].copy()

df_numeric = df_selected.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.', regex=False), errors='coerce'))

df_numeric = df_numeric.dropna(axis=1, how='all')

num_cols = len(df_numeric.columns)

print(f"✅ Відібрано числових ознак: {num_cols} (включно з цільовою метрикою).")

if num_cols < 2:
    print("⚠️ Недостатньо числових ознак для побудови матриці кореляцій (потрібно мінімум 2).")
else:
    print("📊 Побудова матриці кореляцій...")

    corr_matrix = df_numeric.corr()

    clean_labels = [FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').replace('_', ' ').strip()) for col in corr_matrix.columns]

    fig_corr = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=clean_labels,
        y=clean_labels,
        colorscale='RdBu_r',
        zmin=-1, zmax=1,
        texttemplate="%{z:.2f}",
        textfont={"size": 13},
        hovertemplate="<b>Ознака X:</b> %{x}<br><b>Ознака Y:</b> %{y}<br><b>Кореляція:</b> %{z:.4f}<extra></extra>"
    ))

    dynamic_size = max(600, num_cols * 80 + 200)

    fig_corr.update_layout(
        title_text="🔥 Матриця кореляцій Пірсона (Взаємозв'язок факторів)",
        title_x=0.5,
        width=dynamic_size,
        height=dynamic_size,
        template=PLOT_TEMPLATE,
        xaxis=dict(tickangle=-45),
        margin=dict(b=120)
    )

    print("\nКрасивий Вивід:")
    fig_corr.show()

🔍 Відбір числових ознак для аналізу...
✅ Відібрано числових ознак: 7 (включно з цільовою метрикою).
📊 Побудова матриці кореляцій...

Красивий Вивід:


**7. Зробити висновок:**

### Аналіз наявності та сили лінійного зв'язку

Для оцінки взаємозв'язків ми побудували теплову карту на основі **коефіцієнта кореляції Пірсона ($r$)**, який обчислюється за формулою:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

де $\bar{x}$ та $\bar{y}$ — середні значення відповідних ознак. Цей коефіцієнт варіюється в межах $[-1; 1]$, де $1$ — ідеальна пряма залежність, $0$ — відсутність лінійного зв'язку, а $-1$ — ідеальна зворотна залежність. 

Аналізуючи отриману матрицю, можна зробити такі висновки щодо відібраних числових ознак:

1. **Сильний позитивний зв'язок із цільовою метрикою:** Найбільший вплив на Індекс щастя (`Happiness Score`) мають економічний фактор `Economy (GDP per Capita) [ВВП на душу населення]` та соціально-медичний фактор `Health (Life Expectancy) [Тривалість здорового життя]`. Коефіцієнт кореляції для них перевищує 0.75, що вказує на виражену пряму лінійну залежність.
2. **Внутрішня мультиколінеарність (Зв'язок між ознаками):** Спостерігається дуже сильний взаємозв'язок (кореляція ~0.8) між самим ВВП та тривалістю життя. Це логічно з точки зору домену (багатші країни мають кращу медицину). 
3. **Слабкі та помірні зв'язки:** Ознака `Trust (Government Corruption) [Сприйняття корупції]` демонструє значно слабший зв'язок як з індексом щастя, так і з іншими метриками (кореляція в діапазоні 0.2 - 0.4). Це свідчить про те, що цей фактор є більш незалежним і додає системі унікальної дисперсії.
4. **Вплив на подальшу кластеризацію:** Наявність сильної мультиколінеарності означає, що хмара даних у багатовимірному просторі має форму витягнутого еліпсоїда, а не ідеальної сфери. Саме тому використання алгоритму Gaussian Mixture Model (GMM) з параметром `covariance_type='full'` є архітектурно правильним рішенням — цей алгоритм здатен адаптуватися до таких лінійних зв'язків та коректно розділити простір.

**8. Відобразити розподіл:**

In [12]:
print(f"🌍 Побудова теплової мапи для цільової ознаки '{TARGET_METRIC}'...\n")

df[TARGET_METRIC] = pd.to_numeric(df[TARGET_METRIC].astype(str).str.replace(',', '.', regex=False), errors='coerce')

def get_iso3_code(country_name):
    if country_name in MANUAL_ISO_MAPPING:
        return MANUAL_ISO_MAPPING[country_name]
    try:
        return pycountry.countries.search_fuzzy(country_name)[0].alpha_3
    except:
        return None

df['ISO_Code'] = df[COUNTRY_COL].apply(get_iso3_code)

missing_mapping = df[df['ISO_Code'].isna()][COUNTRY_COL].unique()
if len(missing_mapping) > 0:
    print(f"⚠️ Увага! Додайте ці країни у MANUAL_ISO_MAPPING: {missing_mapping}")
else:
    print("✅ Усі країни набору даних успішно перетворено в ISO-3!")

dataset_iso = set(df['ISO_Code'].dropna())
world_iso_dict = {c.alpha_3: c.name for c in pycountry.countries}

total_world = len(world_iso_dict)
total_dataset = len(dataset_iso)
missing_iso = set(world_iso_dict.keys()) - dataset_iso
missing_names = sorted([world_iso_dict[iso] for iso in missing_iso])

print(f"📊 Покриття даних: {total_dataset} країн у наборі із {total_world} офіційних територій (ISO 3166-1).")
print(f"   Відсоток покриття світу: {(total_dataset / total_world * 100):.1f}%\n")

fig_original_map = px.choropleth(
    df,
    locations='ISO_Code',            
    color=TARGET_METRIC,               
    locationmode='ISO-3',            
    color_continuous_scale=COLOR_SCALE_HAPPINESS, 
    hover_name=COUNTRY_COL,
    labels={TARGET_METRIC: "Індекс щастя", 'ISO_Code': "Код ISO"} 
)

fig_original_map.update_traces(
    hovertemplate="<b>%{hovertext}</b><br><br>" +
                  "📌 Код країни: <b>%{location}</b><br>" +
                  "📊 Рівень щастя: <b>%{z:.3f}</b><br>" +
                  "<extra></extra>"
)

fig_original_map.update_layout(
    title_text=f"🗺️ Happiness Index {TARGET_YEAR} (Оригінальні дані)",
    title_x=0.5,
    template=PLOT_TEMPLATE,            
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor="rgba(255, 255, 255, 0.2)",
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)',
        lakecolor='#111111',
        landcolor='#222222'
    ),
    width=1200, height=700
)

print("Красивий Вивід - Оригінальна мапа щастя:")
fig_original_map.show()

print("Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):")
top_bottom_df = pd.concat([
    df[[COUNTRY_COL, TARGET_METRIC]].nlargest(5, TARGET_METRIC),
    df[[COUNTRY_COL, TARGET_METRIC]].nsmallest(5, TARGET_METRIC).sort_values(by=TARGET_METRIC, ascending=False)
])

display(top_bottom_df.style.background_gradient(cmap=DESCRIBE_CMAP, subset=[TARGET_METRIC])\
        .set_properties(**TABLE_PROPS).format({TARGET_METRIC: "{:.4f}"}))

print(f"🛰️ Відсутні країни та території у звіті за {TARGET_YEAR} рік ({len(missing_names)} шт.):")
wrapped_missing = textwrap.fill(", ".join(missing_names), width=120)
print(wrapped_missing)

🌍 Побудова теплової мапи для цільової ознаки 'Happiness.Score'...

✅ Усі країни набору даних успішно перетворено в ISO-3!
📊 Покриття даних: 153 країн у наборі із 249 офіційних територій (ISO 3166-1).
   Відсоток покриття світу: 61.4%

Красивий Вивід - Оригінальна мапа щастя:


Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):


,Country,Happiness.Score
0,Norway,7.5370
1,Denmark,7.5220
2,Iceland,7.5040
3,Switzerland,7.4940
4,Finland,7.4690
150,Rwanda,3.4710
151,Syria,3.4620
152,Tanzania,3.3490
153,Burundi,2.9050
154,Central African Republic,2.6930


🛰️ Відсутні країни та території у звіті за 2017 рік (97 шт.):
American Samoa, Andorra, Anguilla, Antarctica, Antigua and Barbuda, Aruba, Bahamas, Barbados, Bermuda, Bonaire, Sint
Eustatius and Saba, Bouvet Island, British Indian Ocean Territory, Brunei Darussalam, Cabo Verde, Cayman Islands,
Christmas Island, Cocos (Keeling) Islands, Comoros, Cook Islands, Cuba, Curaçao, Djibouti, Dominica, Equatorial Guinea,
Eritrea, Eswatini, Falkland Islands (Malvinas), Faroe Islands, Fiji, French Guiana, French Polynesia, French Southern
Territories, Gambia, Gibraltar, Greenland, Grenada, Guadeloupe, Guam, Guernsey, Guinea-Bissau, Guyana, Heard Island and
McDonald Islands, Holy See (Vatican City State), Isle of Man, Jersey, Kiribati, Korea, Democratic People's Republic of,
Lao People's Democratic Republic, Liechtenstein, Macao, Maldives, Marshall Islands, Martinique, Mayotte, Micronesia,
Federated States of, Monaco, Montserrat, Nauru, New Caledonia, Niger, Niue, Norfolk Island, Northern Mariana Isl

**9. Застосувати стандартизацію даних:**

In [13]:
print(f"⚖️ Масштабування ознак через функцію data_scale() (Метод: {SCALER_TYPE.upper()})...")

def data_scale(data, scaler_type='minmax'):
    if scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif scaler_type == 'std':
        scaler = StandardScaler()
    elif scaler_type == 'norm':
        scaler = Normalizer()
    else:
        raise ValueError("Невідомий тип скейлера!")

    scaler.fit(data)
    res = scaler.transform(data)

    return res, scaler

for col in FEATURES_FULL:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

X_raw = df[FEATURES_FULL].copy()
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)
X_features = pd.DataFrame(X_imputed, columns=FEATURES_FULL, index=df.index)

data_scaled, fitted_scaler = data_scale(X_features, scaler_type=SCALER_TYPE)
df_scaled = pd.DataFrame(data_scaled, columns=FEATURES_FULL, index=df.index)

print("✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)\n")

f1 = FEATURES_FULL[0]
f2 = FEATURES_FULL[1] if len(FEATURES_FULL) > 1 else FEATURES_FULL[0]

f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1))
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2))

SCALER_NAMES = {
    'minmax': 'MinMaxScaler (від 0 до 1)',
    'std': 'StandardScaler (Z-score центрування)',
    'norm': 'Normalizer (Векторна нормалізація)'
}
scaler_display_name = SCALER_NAMES.get(SCALER_TYPE, SCALER_TYPE.upper())

fig_scale = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"Оригінальні дані", f"Відмасштабовано: {scaler_display_name}"),
    horizontal_spacing=0.1
)

fig_scale.add_trace(go.Scatter(
    x=X_features[f1], y=X_features[f2], mode='markers',
    marker=dict(color='#00c3ff', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>" + 
                  f"{f1_ua}: <b>%{{x:.3f}}</b><br>" + 
                  f"{f2_ua}: <b>%{{y:.3f}}</b><extra></extra>"
), row=1, col=1)

fig_scale.add_trace(go.Scatter(
    x=df_scaled[f1], y=df_scaled[f2], mode='markers',
    marker=dict(color='#ccff00', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>" + 
                  f"Масштаб. {f1_ua}: <b>%{{x:.3f}}</b><br>" + 
                  f"Масштаб. {f2_ua}: <b>%{{y:.3f}}</b><extra></extra>"
), row=1, col=2)

fig_scale.update_xaxes(title_text=f1_ua, showgrid=True, gridcolor='#333', row=1, col=1)
fig_scale.update_yaxes(title_text=f2_ua, showgrid=True, gridcolor='#333', row=1, col=1)

fig_scale.update_xaxes(title_text=f"Scaled: {f1_ua}", showgrid=True, gridcolor='#333', row=1, col=2)
fig_scale.update_yaxes(title_text=f"Scaled: {f2_ua}", showgrid=True, gridcolor='#333', row=1, col=2)

fig_scale.update_layout(
    title_text="✨ Трансформація простору: Оригінальні vs Відмасштабовані дані",
    title_x=0.5, width=1500, height=550, template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=80)
)

fig_scale.add_annotation(
    x=0.5, y=-0.20, xref="paper", yref="paper",
    text="💡 Зверніть увагу на осі: форма хмари точок зберігається, але координати стиснуті алгоритмом для потреб машинного навчання.",
    showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
)

print("Красивий Вивід - Візуалізація ефекту масштабування:")
fig_scale.show()

print("\nТехнічний Вивід - Перші 5 рядків відмасштабованих ознак (Без пропусків):")
display(df_scaled.head().style.background_gradient(cmap='Purples').set_properties(**TABLE_PROPS).format("{:.4f}"))

⚖️ Масштабування ознак через функцію data_scale() (Метод: STD)...
✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)

Красивий Вивід - Візуалізація ефекту масштабування:



Технічний Вивід - Перші 5 рядків відмасштабованих ознак (Без пропусків):


,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.
0,1.5062,1.2036,1.0382,1.5158,0.8570,1.9031
1,1.1865,1.2650,1.0208,1.4529,0.8069,2.7400
2,1.1823,1.4727,1.1943,1.4606,1.7020,0.3001
3,1.3834,1.1456,1.2983,1.4132,0.3250,2.4068
4,1.0940,1.2271,1.0910,1.3990,-0.0104,2.5608


**10. Відобразити статистики:**

In [14]:
print("📊 Технічний аудит та порівняння описових статистик...\n")

fig_stats = make_subplots(
    rows=2, cols=1,
    subplot_titles=("1. Розподіл ОРИГІНАЛЬНИХ ознак (Різні масштаби та дисперсії)", f"2. Розподіл ВІДМАСШТАБОВАНИХ ознак ({SCALER_TYPE.upper()})"),
    vertical_spacing=0.12
)

colors = px.colors.qualitative.Pastel

for i, col in enumerate(FEATURES_FULL):
    clean_name = FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').replace('_', ' ').strip())

    fig_stats.add_trace(go.Violin(
        x=X_features[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True, meanline_visible=True, points='outliers',
        hovertemplate=f"<b>{clean_name}</b><br>Значення: %{{x:.3f}}<extra></extra>"
    ), row=1, col=1)

for i, col in enumerate(FEATURES_FULL):
    clean_name = FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').replace('_', ' ').strip())
    
    fig_stats.add_trace(go.Violin(
        x=df_scaled[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True, meanline_visible=True, points='outliers',
        hovertemplate=f"<b>{clean_name} (Scaled)</b><br>Значення: %{{x:.3f}}<extra></extra>"
    ), row=2, col=1)

fig_stats.update_xaxes(title_text="Оригінальні значення", showgrid=True, gridcolor='#333', row=1, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=1, col=1)

fig_stats.update_xaxes(title_text="Відмасштабовані значення", showgrid=True, gridcolor='#333', row=2, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=2, col=1)

dynamic_violin_height = max(500, len(FEATURES_FULL) * 120 + 250)

fig_stats.update_layout(
    title_text="🎻 'Скрипкові діаграми' (Violin Plots): Аналіз щільності та квартилів",
    title_x=0.5, 
    width=1500, height=dynamic_violin_height,
    template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=130, l=150, t=80)
)

fig_stats.add_annotation(
    x=0.5, y=-0.13, xref="paper", yref="paper",
    text="💡 <b>Violin Plot</b> поєднує Boxplot (всередині) та хвилю щільності (зовні). Товщина 'скрипки' показує, де сконцентровано найбільше країн.<br>Зверніть увагу, як нижній графік вирівняв дисперсію всіх ознак, підготувавши їх до GMM!",
    showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
)

print("Красивий Вивід:")
fig_stats.show()

print("Технічний Вивід - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):")
display(X_features.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("\nТехнічний Вивід - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):")
display(df_scaled.describe().T.style.background_gradient(cmap='Purples').format("{:.4f}"))

📊 Технічний аудит та порівняння описових статистик...

Красивий Вивід:


Технічний Вивід - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0000,0.9847,0.4208,0.0000,0.6634,1.0646,1.3180,1.8708
Family,155.0000,1.1889,0.2873,0.0000,1.0426,1.2539,1.4143,1.6106
Health..Life.Expectancy.,155.0000,0.5513,0.2371,0.0000,0.3699,0.6060,0.7230,0.9495
Freedom,155.0000,0.4088,0.1500,0.0000,0.3037,0.4375,0.5166,0.6582
Generosity,155.0000,0.2469,0.1348,0.0000,0.1541,0.2315,0.3238,0.8381
Trust..Government.Corruption.,155.0000,0.1231,0.1017,0.0000,0.0573,0.0898,0.1533,0.4643



Технічний Вивід - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0000,-0.0000,1.0032,-2.3477,-0.7661,0.1904,0.7947,2.1125
Family,155.0000,0.0000,1.0032,-4.1521,-0.5108,0.2271,0.7873,1.4727
Health..Life.Expectancy.,155.0000,-0.0000,1.0032,-2.3332,-0.7680,0.2315,0.7265,1.6849
Freedom,155.0000,0.0000,1.0032,-2.7341,-0.7030,0.1917,0.7208,1.6685
Generosity,155.0000,0.0000,1.0032,-1.8377,-0.6906,-0.1142,0.5722,4.4006
Trust..Government.Corruption.,155.0000,-0.0000,1.0032,-1.2150,-0.6498,-0.3284,0.2978,3.3670


**10.1. Висновок до Кроку 10:**

### Аналіз стандартизованих статистик та щільності розподілу

Порівнюючи описові статистики (`describe()`) та їх візуалізацію через Скрипкові діаграми (Violin Plots) для оригінального та відмасштабованого наборів даних, можна зробити такі математичні висновки:

1. **Трансформація простору (Feature Scaling):** В оригінальному наборі ознаки мали кардинально різні математичні діапазони. Після застосування нашого скейлера всі вектори ознак $x$ були лінійно трансформовані у новий простір $x'$. Наприклад, при стандартизації (StandardScaler) це досягається зміщенням середнього до нуля та нормуванням дисперсії до одиниці:
    $$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
    Це жорстко вписало всі ознаки в єдиний стандартизований масштаб, що яскраво видно по вирівняних "скрипках" на нижньому графіку. 

2. **Уніфікація статистичної ваги:** Завдяки масштабуванню не лише медіани, але й дисперсії (ширина хвилі розподілу) були збалансовані. Це означає, що відтепер жодна ознака не зможе штучно "домінувати" в алгоритмі просто за рахунок більших абсолютних чисел (наприклад, макроекономіка більше не задавить соціальні фактори).

3. **Математична стабільність для GMM:** Скрипкові діаграми наочно показують оцінку щільності (Kernel Density). Наявність "потовщень" вказує на скупчення країн. Алгоритм GMM шукає такі згущення, щоб описати їх багатовимірним нормальним розподілом:
    $$\mathcal{N}(x | \mu, \Sigma) = \frac{1}{\sqrt{(2\pi)^D |\Sigma|}} \exp\left(-\frac{1}{2}(x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$
    Де $\Sigma$ — матриця коваріації, а $D$ — розмірність простору. Якби ми не відмасштабували дані, детермінант $|\Sigma|$ та обернена матриця $\Sigma^{-1}$ були б екстремально спотворені ознакою з найбільшою дисперсією. Масштабування гарантує, що багатовимірні еліпси Гаусса формуватимуться на основі реальної геометрії даних, а не через різницю в одиницях виміру.

**11. Побудувати модель кластеризації:**

In [15]:
print(f"🧠 Навчання моделі кластеризації (GaussianMixture)...")
print("🧹 Перевірка та очищення даних від пропущених значень (NaN)...")

missing_values_count = df_scaled[FEATURES_FULL].isna().sum().sum()

if missing_values_count > 0:
    print(f"   ⚠️ Знайдено {missing_values_count} пропущених значень у числових ознаках.")
    print("   🛠️ Застосовуємо імп'ютацію (заповнення медіаною)...")

    imputer = SimpleImputer(strategy='median')

    df_scaled[FEATURES_FULL] = imputer.fit_transform(df_scaled[FEATURES_FULL])
    df[FEATURES_FULL] = imputer.fit_transform(df[FEATURES_FULL])

    print("   ✅ Пропущені значення успішно заповнено!")
else:
    print("   ✅ Пропущених значень не знайдено. Дані чисті.")

X_train = df_scaled[FEATURES_FULL]

gmm_brain = GaussianMixture(
    n_components=N_CLUSTERS, covariance_type=COVARIANCE_TYPE, 
    n_init=N_INIT, random_state=RANDOM_STATE
)
cluster_labels = gmm_brain.fit_predict(X_train)

df['Cluster'] = cluster_labels
cluster_means = df.groupby('Cluster')[TARGET_METRIC].mean().sort_values()
cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(cluster_means.index)}
df['Cluster'] = df['Cluster'].map(cluster_mapping)

df['Cluster_Name'] = df['Cluster'].map(lambda x: CLUSTER_LABELS.get(x, f"Кластер {x+1}"))

actual_iters = gmm_brain.n_iter_
print(f"✅ Навчання завершено! Алгоритм зійшовся за {actual_iters} ітерацій.")

if len(FEATURES_FULL) < 2:
    print("⚠️ Для 2D-візуалізації GMM та PCA потрібно мінімум 2 ознаки. Побудову графіків пропущено.")
else:
    f1, f2 = FEATURES_FULL[0], FEATURES_FULL[1]
    f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1).replace('.', ' ').strip())
    f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2).replace('.', ' ').strip())

    print("\nКрасивий Вивід:")

    def get_dynamic_legend(X_coords, x_min, x_max, y_min, y_max):
        x_mid = (x_min + x_max) / 2
        y_mid = (y_min + y_max) / 2

        top_left = np.sum((X_coords[:, 0] <= x_mid) & (X_coords[:, 1] >= y_mid))
        top_right = np.sum((X_coords[:, 0] > x_mid) & (X_coords[:, 1] >= y_mid))
        bottom_left = np.sum((X_coords[:, 0] <= x_mid) & (X_coords[:, 1] < y_mid))
        bottom_right = np.sum((X_coords[:, 0] > x_mid) & (X_coords[:, 1] < y_mid))

        min_points = min(top_left, top_right, bottom_left, bottom_right)

        if min_points == top_left: return dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
        elif min_points == top_right: return dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
        elif min_points == bottom_left: return dict(yanchor="bottom", y=0.01, xanchor="left", x=0.01)
        else: return dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99)

    def extract_2d_cov(gmm_model, k_idx, d1, d2):
        if gmm_model.covariance_type == 'full':
            return gmm_model.covariances_[k_idx][np.ix_([d1, d2], [d1, d2])]
        elif gmm_model.covariance_type == 'tied':
            return gmm_model.covariances_[np.ix_([d1, d2], [d1, d2])]
        elif gmm_model.covariance_type == 'diag':
            return np.diag([gmm_model.covariances_[k_idx, d1], gmm_model.covariances_[k_idx, d2]])
        elif gmm_model.covariance_type == 'spherical':
            return np.diag([gmm_model.covariances_[k_idx], gmm_model.covariances_[k_idx]])

    print(f"\n🎬 1/3: Генерація динамічної анімації навчання...")
    X_anim = df_scaled[[f1, f2]].values 
    frames = []
    em_steps = actual_iters

    x_min = np.floor(X_anim[:, 0].min() / 0.5) * 0.5
    x_max = np.ceil(X_anim[:, 0].max() / 0.5) * 0.5
    y_min = np.floor(X_anim[:, 1].min() / 0.5) * 0.5
    y_max = np.ceil(X_anim[:, 1].max() / 0.5) * 0.5

    for i in range(1, em_steps + 1):
        gmm_anim = GaussianMixture(
            n_components=N_CLUSTERS, covariance_type=COVARIANCE_TYPE, 
            max_iter=i, n_init=1, init_params='kmeans', random_state=RANDOM_STATE
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            gmm_anim.fit(X_anim)

        current_means = gmm_anim.means_

        frame_traces = [
            go.Scatter(x=X_anim[:, 0], y=X_anim[:, 1], mode='markers', marker=dict(color='#888', size=6, opacity=0.3), showlegend=False)
        ]

        for k in range(N_CLUSTERS):
            safe_color = COLOR_PALETTE_FULL[k % len(COLOR_PALETTE_FULL)]
            frame_traces.append(go.Scatter(
                x=[current_means[k, 0]], y=[current_means[k, 1]], 
                mode='markers+text', text=[f"Гаусс {k+1}"],
                marker=dict(color=safe_color, size=22, line=dict(width=3, color='white')), showlegend=False))

        frames.append(go.Frame(data=frame_traces, name=str(i)))

    sliders = [{
        "pad": {"b": 10, "t": 60}, "len": 0.9, "x": 0.1, "y": 0,
        "currentvalue": {"font": {"size": 16}, "prefix": "Ітерація: ", "visible": True, "xanchor": "right"},
        "steps": [{"args": [[f.name], {"frame": {"duration": 400, "redraw": True}, "mode": "immediate", "transition": {"duration": 200}}],
                   "label": str(k+1), "method": "animate"} for k, f in enumerate(frames)]
    }]

    fig_anim = go.Figure(data=frames[0].data, frames=frames)
    fig_anim.update_layout(
        title_text=f"🎬 1. Процес навчання: Динаміка EM-алгоритму (2D Зріз)",
        title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=600,
        xaxis=dict(title=f"📐 Scaled: {f1_ua}", range=[x_min, x_max]),
        yaxis=dict(title=f"🩺 Scaled: {f2_ua}", range=[y_min, y_max]),
        updatemenus=[dict(type="buttons", y=-0.1, x=0.05, buttons=[
            dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 400, "redraw": True}, "fromcurrent": True, "transition": {"duration": 200}}]),
            dict(label="⏸ Pause", method="animate", args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])])],
        sliders=sliders
    )

    fig_anim.show()

    print("\n📐 2/3: Побудова контурних карт коваріації...")
    f1_idx, f2_idx = 0, 1

    x_range, y_range = np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100)
    XX, YY = np.meshgrid(x_range, y_range)
    pos = np.dstack((XX, YY))

    fig_contours = go.Figure()
    fig_contours.add_trace(go.Histogram2dContour(x=X_anim[:, 0], y=X_anim[:, 1], colorscale='Greys', opacity=0.3, showscale=False))

    for i in range(N_CLUSTERS):
        mean = gmm_brain.means_[i, [f1_idx, f2_idx]]
        cov = extract_2d_cov(gmm_brain, i, f1_idx, f2_idx)
        rv = multivariate_normal(mean=mean, cov=cov)
        Z = rv.pdf(pos)

        color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
        name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")

        fig_contours.add_trace(go.Contour(
            x=x_range, y=y_range, z=Z, colorscale=[[0, 'rgba(0,0,0,0)'], [1, color]],
            showscale=False, contours=dict(start=0.01, coloring='lines'), line=dict(width=1), hoverinfo='skip'))

    for i in range(N_CLUSTERS):
        real_indices = (cluster_labels == cluster_mapping.get(i, i))
        real_data = df[real_indices]

        hover_texts = [f"<b>{row[COUNTRY_COL]}</b><br>{f1_ua} (Raw): {row[f1]:.2f}<br>{f2_ua} (Raw): {row[f2]:.2f}" for _, row in real_data.iterrows()]
        color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
        name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")

        fig_contours.add_trace(go.Scatter(
            x=X_anim[cluster_labels == cluster_mapping.get(i, i), 0], y=X_anim[cluster_labels == cluster_mapping.get(i, i), 1],
            mode='markers', name=name, marker=dict(color=color, size=7, opacity=0.8, line=dict(width=1, color='black')),
            hovertext=hover_texts, hoverinfo="text",
        ))

    padding_x = (x_max - x_min) * 0.10
    padding_y = (y_max - y_min) * 0.10

    focus_x0 = np.min(gmm_brain.means_[:, f1_idx]) - padding_x
    focus_x1 = np.max(gmm_brain.means_[:, f1_idx]) + padding_x
    focus_y0 = np.min(gmm_brain.means_[:, f2_idx]) - padding_y
    focus_y1 = np.max(gmm_brain.means_[:, f2_idx]) + padding_y

    fig_contours.add_shape(
        type="rect",
        x0=focus_x0, y0=focus_y0, x1=focus_x1, y1=focus_y1,
        line=dict(color="#00ffcc", width=2, dash="dashdot"),
        fillcolor="rgba(0,0,0,0)"
    )

    fig_contours.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        name="Фокус центрів",
        line=dict(color="#00ffcc", width=2, dash="dashdot")
    ))

    fig_contours.update_layout(
        title_text=f"📐 2. Геометрія: Гаусівські контурні карти коваріації ({COVARIANCE_TYPE.upper()})",
        title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=650,
        xaxis=dict(title=f"Scaled: {f1_ua}", showgrid=False, range=[x_min, x_max]),
        yaxis=dict(title=f"Scaled: {f2_ua}", showgrid=False, range=[y_min, y_max]),
        legend=get_dynamic_legend(X_anim, x_min, x_max, y_min, y_max)
    )

    fig_contours.show()

    print("\n🌌 3/3: Стиснення простору через PCA...")
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_train)

    fig_pca = go.Figure()

    pca_x_min = np.floor(X_pca[:, 0].min() / 0.5) * 0.5
    pca_x_max = np.ceil(X_pca[:, 0].max() / 0.5) * 0.5
    pca_y_min = np.floor(X_pca[:, 1].min() / 0.5) * 0.5
    pca_y_max = np.ceil(X_pca[:, 1].max() / 0.5) * 0.5

    xx, yy = np.meshgrid(np.linspace(pca_x_min, pca_x_max, 100), np.linspace(pca_y_min, pca_y_max, 100))
    pos = np.dstack((xx, yy))

    for i in range(N_CLUSTERS):
        cluster_points = X_pca[df['Cluster'] == i]

        if len(cluster_points) > 1:
            pca_mean = np.mean(cluster_points, axis=0)
            pca_cov = np.cov(cluster_points, rowvar=False)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                rv = multivariate_normal(mean=pca_mean, cov=pca_cov, allow_singular=True) 
                Z = rv.pdf(pos)

            color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
            name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")

            fig_pca.add_trace(go.Contour(
                x=np.linspace(pca_x_min, pca_x_max, 100), y=np.linspace(pca_y_min, pca_y_max, 100), z=Z,
                showscale=False, contours=dict(start=0.1, coloring='lines'), line=dict(width=1, color=color), hoverinfo='skip'))

        real_indices = (cluster_labels == cluster_mapping.get(i, i))
        real_data = df[real_indices]
        hover_texts_pca = [f"<b>{row[COUNTRY_COL]}</b><br>PCA 1: {X_pca[idx, 0]:.2f}<br>PCA 2: {X_pca[idx, 1]:.2f}" for idx, row in real_data.reset_index(drop=True).iterrows()]
        
        color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
        name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")

        fig_pca.add_trace(go.Scatter(
            x=cluster_points[:, 0], y=cluster_points[:, 1], mode='markers', name=name,
            marker=dict(color=color, size=9, opacity=0.8, line=dict(width=1, color='black')),
            hovertext=hover_texts_pca, hoverinfo="text"
        ))

    variance_explained = sum(pca.explained_variance_ratio_) * 100
    fig_pca.update_layout(
        title_text=f"🌌 3. Фінальний результат: Абстрактна 2D-Проєкція (PCA). Збережено {variance_explained:.2f}% інформації",
        title_x=0.5, template=PLOT_TEMPLATE, width=1050, height=700,
        xaxis=dict(title=f"← Головна компонента 1 →", showgrid=True, gridcolor='#333', range=[pca_x_min, pca_x_max]),
        yaxis=dict(title=f"← Головна компонента 2 →", showgrid=True, gridcolor='#333', range=[pca_y_min, pca_y_max]),
        legend=get_dynamic_legend(X_pca, pca_x_min, pca_x_max, pca_y_min, pca_y_max)
    )

    fig_pca.show()

🧠 Навчання моделі кластеризації (GaussianMixture)...
🧹 Перевірка та очищення даних від пропущених значень (NaN)...
   ✅ Пропущених значень не знайдено. Дані чисті.
✅ Навчання завершено! Алгоритм зійшовся за 9 ітерацій.

Красивий Вивід:

🎬 1/3: Генерація динамічної анімації навчання...



📐 2/3: Побудова контурних карт коваріації...



🌌 3/3: Стиснення простору через PCA...


**11.5.\*\* Експорт навченої моделі ШІ:**

In [16]:
print(f"🪬 Експорт моделі (Збереження 'Мозку' GMM для {TARGET_YEAR} року)...\n")

if 'gmm_brain' not in globals() or 'fitted_scaler' not in globals():
    raise SystemExit("\n❌ Критична Помилка Архітектури: Розсинхронізація стану!\n"
                     "   У пам'яті зараз відсутня натренована модель GMM або Скейлер.\n"
                     "   👉 РІШЕННЯ: Запустіть попередній блок 'Навчання моделі' ще раз!")

actual_trained_clusters = gmm_brain.n_components
current_model_id = id(gmm_brain)
last_saved_id = globals().get('_LAST_SAVED_MODEL_ID')
last_saved_time = globals().get('_LAST_SAVED_TIMESTAMP_HUMAN')

model_exists = os.path.exists(MODEL_PATH)

if actual_trained_clusters != N_CLUSTERS:
    print("\n❌ Критична Помилка Архітектури: Розсинхронізація стану!")
    print(f"   Ти змінив константу кластерів на [{N_CLUSTERS}].")
    print(f"   Але в пам'яті зараз висить 'Мозок', натренований на [{actual_trained_clusters}] кластерів.")
    print("   👉 РІШЕННЯ: Запусти блок 'Навчання моделі' ще раз, щоб перенавчити математику, а потім повертайся сюди!")
elif current_model_id == last_saved_id and model_exists:
    print(f"   ♻️ Ця конкретна модель вже була збережена на диск (Час фіксації: {last_saved_time}).")
    print("   💬 Вікно збереження автоматично пропущено, щоб уникнути дублювання.")
else:
    timestamp_now = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    timestamp_human = pd.Timestamp.now().strftime("%d.%m.%Y %H:%M:%S")

    default_save = globals().get('DEFAULT_SAVE_MODEL', True)
    default_action_text = '💚 Зберігати' if default_save else '💛 Не зберігати'

    if model_exists:
        base_msg = f"⚠️ Капсула моделі '{MODEL_PATH}' вже існує. Перезаписати її?"
    else:
        base_msg = "❓ Зберегти цю навчену політику (Мозок) на Диск?"

    prompt_msg = f"   {base_msg} (Так/Ні) [За замовчуванням: {default_action_text}]: "
    user_input = input(prompt_msg).strip().lower()

    if user_input in ['y', 'yes', 'т', 'так']:
        should_save = True
    elif user_input in ['n', 'no', 'н', 'ні']:
        should_save = False
    elif user_input == '': 
        should_save = default_save
        print(f"   ↩️ Натиснуто Enter. Використовуємо значення за замовчуванням: {default_action_text}")
    else:
        should_save = default_save
        print(f"   ❓ Відповідь не розпізнана. Використовуємо значення за замовчуванням: {default_action_text}")

    if should_save:
        try:
            model_dir = os.path.dirname(MODEL_PATH)
            if model_dir:
                os.makedirs(model_dir, exist_ok=True)

            if model_exists:
                try:
                    file_mtime = os.path.getmtime(MODEL_PATH)
                    old_time_str = pd.Timestamp(file_mtime, unit='s').strftime("%Y%m%d_%H%M%S")
                    
                    backup_name = f"GMM_backup_{old_time_str}.pkl"
                    backup_path = os.path.join(model_dir, backup_name) if model_dir else backup_name

                    shutil.copy2(MODEL_PATH, backup_path)
                    print(f"      📥 Створено резервну копію старої моделі: {backup_name}")
                except Exception as backup_err:
                    print(f"      ⚠️ Не вдалося створити резервну копію: {backup_err}")

            dynamic_labels = {i: CLUSTER_LABELS.get(i, f"Кластер {i+1}") for i in range(actual_trained_clusters)}
            dynamic_colors = [COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)] for i in range(actual_trained_clusters)]

            safe_cluster_mapping = globals().get('cluster_mapping', {i: i for i in range(actual_trained_clusters)})

            model_payload = {
                "algorithm": "Gaussian Mixture Model (GMM)",
                "brain": gmm_brain,
                "scaler": fitted_scaler,
                "scaler_type": SCALER_TYPE,
                "target_year": TARGET_YEAR,
                "features": FEATURES_FULL,
                "cluster_labels": dynamic_labels,
                "cluster_mapping": safe_cluster_mapping,
                "hyperparameters": {
                    "n_clusters": N_CLUSTERS,
                    "covariance_type": COVARIANCE_TYPE,
                    "init_params": GMM_INIT_PARAMS,
                    "n_init": N_INIT
                },
                "ui_colors": dynamic_colors,
                "timestamp": timestamp_now
            }

            joblib.dump(model_payload, MODEL_PATH)

            df.to_csv(EXPORT_CSV_PATH, index=False, encoding='utf-8-sig')

            globals()['_LAST_SAVED_MODEL_ID'] = current_model_id
            globals()['_LAST_SAVED_TIMESTAMP_HUMAN'] = timestamp_human

            file_size_kb = os.path.getsize(MODEL_PATH) / 1024
            csv_size_kb = os.path.getsize(EXPORT_CSV_PATH) / 1024

            print("   🎭 Початок збереження у файл...")
            print(f"      📦 Файл: {MODEL_PATH} (Матриці Коваріації + Скейлер + Метадані)")
            print(f"      🎛 Ознаки: {len(FEATURES_FULL)} вимірів (Масштабування: {SCALER_TYPE})")
            print("      👾 Архітектура моделі:")
            print(f"         🧠 Алгоритм: GMM (Expectation-Maximization)")
            print(f"         🧬 Гаусівських розподілів: {actual_trained_clusters}")
            print(f"         🔬 Тип коваріації: {COVARIANCE_TYPE}")
            print(f"         🔄 Ітерацій до збіжності: {gmm_brain.n_iter_}")
            print(f"      🏷 Класи: {', '.join(list(dynamic_labels.values()))}")
            print(f"   ✅ Успіх! Капсула 'Мозку' ШІ надійно збережена ({file_size_kb:.2f} KB) о {timestamp_human}")
            print(f"   ✅ Набір даних з мітками експортовано ({csv_size_kb:.2f} KB)")
            print("   🎨 UI-Палітра та Словники: Успішно запаковані всередину файлу для майбутнього Інференсу!")

        except Exception as e:
            print(f"\n   ❌ Сталася помилка під час збереження: {e}")
    else:
        print("   ⏭️ Збереження пропущено. Файли на диску не змінено...")

🪬 Експорт моделі (Збереження 'Мозку' GMM для 2017 року)...

   ↩️ Натиснуто Enter. Використовуємо значення за замовчуванням: 💚 Зберігати
      📥 Створено резервну копію старої моделі: GMM_backup_20260331_233041.pkl
   🎭 Початок збереження у файл...
      📦 Файл: GMM_Models/gmm_2017_full_kmeans_model.pkl (Матриці Коваріації + Скейлер + Метадані)
      🎛 Ознаки: 6 вимірів (Масштабування: std)
      👾 Архітектура моделі:
         🧠 Алгоритм: GMM (Expectation-Maximization)
         🧬 Гаусівських розподілів: 3
         🔬 Тип коваріації: full
         🔄 Ітерацій до збіжності: 9
      🏷 Класи: Низький рівень, Середній рівень, Високий рівень
   ✅ Успіх! Капсула 'Мозку' ШІ надійно збережена (5.46 KB) о 01.04.2026 01:57:29
   ✅ Набір даних з мітками експортовано (33.71 KB)
   🎨 UI-Палітра та Словники: Успішно запаковані всередину файлу для майбутнього Інференсу!


**12. Побудувати теплову мапу:**

In [17]:
print("🗺️ Ініціалізація картографічного модуля (Choropleth Map)...")

if 'ISO_Code' not in df.columns or 'Cluster_Name' not in df.columns:
    raise SystemExit("❌ Критична помилка: У наборі даних відсутні колонки 'ISO_Code' або 'Cluster_Name'.\n"
                     "   👉 РІШЕННЯ: Переконайтеся, що ви виконали блоки генерації ISO-кодів та навчання GMM.")

print("   ✅ Геодані та мітки кластерів знайдено. Будуємо проєкцію...\n")

df['Global_Rank'] = df[TARGET_METRIC].rank(ascending=False, method='min').astype('Int64')

unique_clusters = sorted(df['Cluster'].dropna().unique())
discrete_color_map = {
    CLUSTER_LABELS.get(i, f"Кластер {i+1}"): COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)] 
    for i in unique_clusters
}

f1 = FEATURES_FULL[0]
f2 = FEATURES_FULL[1] if len(FEATURES_FULL) > 1 else FEATURES_FULL[0]

f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1).replace('.', ' ').replace('_', ' ').strip())
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2).replace('.', ' ').replace('_', ' ').strip())

fig_cluster_map = px.choropleth(
    df,
    locations='ISO_Code',            
    color='Cluster_Name',              
    locationmode='ISO-3',            
    color_discrete_map=discrete_color_map,
    hover_name=COUNTRY_COL,
    labels={'Cluster_Name': "Рівень життя (ШІ)"},
    custom_data=['Cluster_Name', TARGET_METRIC, f1, f2, 'Global_Rank']
)

fig_cluster_map.update_traces(
    hovertemplate="<b>%{hovertext}</b><br><br>" +
                  "🏆 Світовий рейтинг: <b>#%{customdata[4]}</b><br>" +
                  "📌 Код ISO: <b>%{location}</b><br>" +
                  "🤖 Кластер ШІ: <b>%{customdata[0]}</b><br>" +
                  "📊 Оригінальний індекс: <b>%{customdata[1]:.3f}</b><br>" +
                  f"💰 {f1_ua}: <b>%{{customdata[2]:.2f}}</b><br>" +
                  f"🩺 {f2_ua}: <b>%{{customdata[3]:.2f}}</b><br>" +
                  "<extra></extra>"
)

fig_cluster_map.update_layout(
    title_text=f"🌐 Світовий розподіл рівнів життя за версією ШІ (GMM Clusters, {TARGET_YEAR})",
    title_x=0.5,
    title_font=dict(size=20, color="#ffffff"),
    template=PLOT_TEMPLATE,            
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor="rgba(255, 255, 255, 0.3)",
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)',
        lakecolor='#0a0a0a',
        landcolor='#1a1a1a'
    ),
    legend=dict(
        title="Категорії (Кластери)",
        yanchor="bottom", y=0.05, 
        xanchor="left", x=0.05,
        bgcolor="rgba(0,0,0,0.5)",
        bordercolor="#444", borderwidth=1
    ),
    width=1300, height=750,
    margin=dict(l=0, r=0, b=0, t=60)
)

print("Красивий Вивід - Кластерна мапа світу:")
fig_cluster_map.show()

print("\n📊 Технічний Вивід - Аналітика сформованих кластерів:")
cluster_stats = df.groupby('Cluster_Name').agg(
    count=(COUNTRY_COL, 'count'),
    mean_idx=(TARGET_METRIC, 'mean'),
    min_idx=(TARGET_METRIC, 'min'),
    max_idx=(TARGET_METRIC, 'max')
).reset_index().rename(columns={
    'count': 'Кількість країн',
    'mean_idx': 'Середній індекс',
    'min_idx': 'Мін. індекс',
    'max_idx': 'Макс. індекс'
}).sort_values('Середній індекс', ascending=False)

display(cluster_stats.style.background_gradient(cmap='YlGnBu', subset=['Середній індекс'])\
        .set_properties(**TABLE_PROPS)\
        .format({
            'Середній індекс': '{:.3f}', 
            'Мін. індекс': '{:.3f}', 
            'Макс. індекс': '{:.3f}'
        }))

🗺️ Ініціалізація картографічного модуля (Choropleth Map)...
   ✅ Геодані та мітки кластерів знайдено. Будуємо проєкцію...

Красивий Вивід - Кластерна мапа світу:



📊 Технічний Вивід - Аналітика сформованих кластерів:


,Cluster_Name,Кількість країн,Середній індекс,Мін. індекс,Макс. індекс
0,Високий рівень,26,6.877,5.472,7.537
2,Середній рівень,79,5.579,3.349,7.213
1,Низький рівень,50,4.206,2.693,5.971


**12.5. Інтерактивна 3D-модель простору ознак:**

In [18]:
if len(FEATURES_FULL) >= 3:
    print("🌌 Побудова багатовимірної 3D-проєкції кластерів...")

    f_x, f_y, f_z = FEATURES_FULL[0], FEATURES_FULL[1], FEATURES_FULL[2]

    name_x = FEATURE_TRANSLATIONS.get(f_x, str(f_x).replace('.', ' ').strip())
    name_y = FEATURE_TRANSLATIONS.get(f_y, str(f_y).replace('.', ' ').strip())
    name_z = FEATURE_TRANSLATIONS.get(f_z, str(f_z).replace('.', ' ').strip())

    fig_3d = px.scatter_3d(
        df,
        x=f_x, y=f_y, z=f_z,
        color='Cluster_Name',
        color_discrete_map=discrete_color_map,
        hover_name=COUNTRY_COL,
        labels={
            f_x: name_x,
            f_y: name_y,
            f_z: name_z,
            'Cluster_Name': 'Клас'
        },
        opacity=0.85
    )

    fig_3d.update_traces(
        marker=dict(size=6, line=dict(width=1, color='Black')),
        hovertemplate="<b>%{hovertext}</b><br><br>" +
                      f"💰 {name_x}: <b>%{{x:.2f}}</b><br>" +
                      f"🤝 {name_y}: <b>%{{y:.2f}}</b><br>" +
                      f"🩺 {name_z}: <b>%{{z:.2f}}</b><br>" +
                      "<extra></extra>"
    )

    fig_3d.update_layout(
        title_text=f"🌌 3D-Анатомія Кластерів ШІ (Рік: {TARGET_YEAR})",
        title_x=0.5,
        title_font=dict(size=20, color="#ffffff"),
        template=PLOT_TEMPLATE,
        scene=dict(
            xaxis_title=name_x,
            yaxis_title=name_y,
            zaxis_title=name_z,
            xaxis=dict(gridcolor='#333', backgroundcolor='#111'),
            yaxis=dict(gridcolor='#333', backgroundcolor='#111'),
            zaxis=dict(gridcolor='#333', backgroundcolor='#111'),
            bgcolor="#0a0a0a"
        ),
        legend=dict(
            title="Категорії (Кластери)",
            yanchor="top", y=0.9, 
            xanchor="left", x=0.05,
            bgcolor="rgba(0,0,0,0.5)",
            bordercolor="#444", borderwidth=1
        ),
        width=1200, height=800,
        margin=dict(l=0, r=0, b=0, t=60)
    )

    print("Красивий Вивід:")
    fig_3d.show()
else:
    print("⚠️ Для побудови 3D-графіка потрібно мінімум 3 активні ознаки у змінній ACTIVE_DIMENSIONS!")

🌌 Побудова багатовимірної 3D-проєкції кластерів...
Красивий Вивід:


**13. Дослідити вплив:**

In [19]:
print("🔬 Дослідження впливу набору ознак на результати кластеризації...\n")

SAFE_FEATURES_MINI = FEATURES_FULL[:2] if len(FEATURES_FULL) >= 2 else FEATURES_FULL

print(f"   ⚙️ Для експерименту зі зниженою розмірністю автоматично обрано: {SAFE_FEATURES_MINI}\n")

X_train_mini = df_scaled[SAFE_FEATURES_MINI]
gmm_mini = GaussianMixture(
    n_components=N_CLUSTERS, covariance_type=COVARIANCE_TYPE,
    n_init=N_INIT, random_state=RANDOM_STATE
)
df['Cluster_Mini'] = gmm_mini.fit_predict(X_train_mini)

cluster_means_mini = df.groupby('Cluster_Mini')[TARGET_METRIC].mean().sort_values()
cluster_mapping_mini = {old_id: new_id for new_id, old_id in enumerate(cluster_means_mini.index)}
df['Cluster_Mini'] = df['Cluster_Mini'].map(cluster_mapping_mini)

df['Cluster_Mini_Name'] = df['Cluster_Mini'].map(lambda x: CLUSTER_LABELS.get(x, f"Кластер {x+1}"))

ari_score = adjusted_rand_score(df['Cluster'], df['Cluster_Mini'])
changed_df = df[df['Cluster'] != df['Cluster_Mini']].copy()
percent_changed = (len(changed_df) / len(df)) * 100

f1 = SAFE_FEATURES_MINI[0]
f2 = SAFE_FEATURES_MINI[1] if len(SAFE_FEATURES_MINI) > 1 else SAFE_FEATURES_MINI[0]

f1_ua = FEATURE_TRANSLATIONS.get(f1, str(f1).replace('.', ' ').strip())
f2_ua = FEATURE_TRANSLATIONS.get(f2, str(f2).replace('.', ' ').strip())

x_pad = (X_train_mini[f1].max() - X_train_mini[f1].min()) * 0.1
y_pad = (X_train_mini[f2].max() - X_train_mini[f2].min()) * 0.1

x_min, x_max = X_train_mini[f1].min() - x_pad, X_train_mini[f1].max() + x_pad
y_min, y_max = X_train_mini[f2].min() - y_pad, X_train_mini[f2].max() + y_pad

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

grid_df = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=[f1, f2])

Z = gmm_mini.score_samples(grid_df)
Z = Z.reshape(xx.shape)

fig_scatter_combined = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"Оригінал ({len(FEATURES_FULL)} ознак)", f"Еліпси GMM (Тільки {f1_ua} та {f2_ua})"),
    horizontal_spacing=0.1
)

for i in range(N_CLUSTERS):
    cluster_name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")
    color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
    
    orig_data = df[df['Cluster'] == i]
    if not orig_data.empty:
        fig_scatter_combined.add_trace(go.Scatter(
            x=orig_data[f1], y=orig_data[f2], mode='markers', name=cluster_name,
            marker=dict(color=color, size=9, opacity=0.8, line=dict(width=1, color='black')),
            text=orig_data[COUNTRY_COL],
            hovertemplate=f"<b>%{{text}}</b><br>Кластер: {cluster_name}<extra></extra>",
            legendgroup=cluster_name, showlegend=True
        ), row=1, col=1)

fig_scatter_combined.add_trace(go.Contour(
    x=np.linspace(x_min, x_max, 100), y=np.linspace(y_min, y_max, 100), z=Z,
    colorscale='Viridis', opacity=0.3, showscale=False,
    contours=dict(coloring='lines', showlabels=False), line=dict(width=1.5),
    hoverinfo='skip', name="Щільність GMM"
), row=1, col=2)

for i in range(N_CLUSTERS):
    cluster_name = CLUSTER_LABELS.get(i, f"Кластер {i+1}")
    color = COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)]
    mask = df['Cluster_Mini_Name'] == cluster_name

    if mask.sum() > 0:
        fig_scatter_combined.add_trace(go.Scatter(
            x=X_train_mini.loc[mask, f1], y=X_train_mini.loc[mask, f2], mode='markers', name=cluster_name,
            marker=dict(color=color, size=10, opacity=0.9, line=dict(width=1, color='white')),
            text=df.loc[mask, COUNTRY_COL], customdata=df.loc[mask, [f1, f2, TARGET_METRIC]],
            hovertemplate=f"<b>%{{text}}</b><br><br>🤖 Кластер: <b>{cluster_name}</b><br>💰 {f1_ua}: <b>%{{customdata[0]:.2f}}</b><br>🩺 {f2_ua}: <b>%{{customdata[1]:.2f}}</b><br>📊 Індекс: <b>%{{customdata[2]:.3f}}</b><extra></extra>",
            legendgroup=cluster_name, showlegend=False
        ), row=1, col=2)

fig_scatter_combined.update_layout(
    title_text="⚖️ Порівняння просторів: Оригінальний розподіл vs. Ймовірнісні еліпси",
    title_x=0.5, template=PLOT_TEMPLATE, width=1500, height=650,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
)

fig_scatter_combined.update_xaxes(title_text=f1_ua, row=1, col=1)
fig_scatter_combined.update_yaxes(title_text=f2_ua, row=1, col=1)
fig_scatter_combined.update_xaxes(title_text=f"{f1_ua} (Нормал.)", row=1, col=2)
fig_scatter_combined.update_yaxes(title_text=f"{f2_ua} (Нормал.)", row=1, col=2)

print("Красивий Вивід - Порівняльний графік:")
fig_scatter_combined.show()

print("📡 Побудова комплексних аналітичних профілів кластерів...\n")

cols_orig = [TARGET_METRIC] + FEATURES_FULL
cols_mini = [TARGET_METRIC] + FEATURES_MINI

orig_profile = df.groupby('Cluster_Name')[cols_orig].mean()
mini_profile = df.groupby('Cluster_Mini_Name')[cols_mini].mean()

radar_scaler = MinMaxScaler()
orig_profile_scaled = pd.DataFrame(
    radar_scaler.fit_transform(orig_profile), index=orig_profile.index, columns=orig_profile.columns
)
mini_profile_scaled = pd.DataFrame(
    radar_scaler.fit_transform(mini_profile), index=mini_profile.index, columns=mini_profile.columns
)

def make_radar_trace_simple(profile_row, columns, color, name, showleg):
    cols_ua = [FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').strip()) for col in columns]
    values = profile_row[columns].tolist() + [profile_row[columns].tolist()[0]]
    labels = cols_ua + [cols_ua[0]]
    return go.Scatterpolar(
        r=values, theta=labels, fill='toself', name=name, marker_color=color, opacity=0.8,
        legendgroup=name, showlegend=showleg,
        hovertemplate="📊 %{theta}: %{r:.3f}<extra></extra>"
    )

def make_radar_trace_scaled(scaled_row, orig_row, columns, color, name, showleg):
    cols_ua = [FEATURE_TRANSLATIONS.get(col, str(col).replace('.', ' ').strip()) for col in columns]
    values_scaled = scaled_row[columns].tolist() + [scaled_row[columns].tolist()[0]]
    values_orig = orig_row[columns].tolist() + [orig_row[columns].tolist()[0]]
    labels = cols_ua + [cols_ua[0]]
    return go.Scatterpolar(
        r=values_scaled, theta=labels, customdata=values_orig,
        fill='toself', name=name, marker_color=color, opacity=0.8,
        legendgroup=name, showlegend=showleg,
        hovertemplate="📊 <b>%{theta}</b><br>Відносний рівень (0-1): <b>%{r:.2f}</b><br>Абсолютне значення: <b>%{customdata:.3f}</b><extra></extra>"
    )

fig_radar_combined = make_subplots(
    rows=2, cols=2, 
    subplot_titles=(
        "Оригінал (Абсолютні значення)", "Міні-модель (Абсолютні значення)",
        "Оригінал (Нормалізовані 0-1)", "Міні-модель (Нормалізовані 0-1)"
    ),
    specs=[[{'type': 'polar'}, {'type': 'polar'}],
           [{'type': 'polar'}, {'type': 'polar'}]],
    vertical_spacing=0.15
)

draw_order = orig_profile_scaled.mean(axis=1).sort_values(ascending=False).index.tolist()

cluster_to_color = {
    CLUSTER_LABELS.get(i, f"Кластер {i+1}"): COLOR_PALETTE_FULL[i % len(COLOR_PALETTE_FULL)] 
    for i in range(N_CLUSTERS)
}

for cluster_name in draw_order:
    color = cluster_to_color.get(cluster_name, '#ffffff')

    if cluster_name in orig_profile.index:
        fig_radar_combined.add_trace(make_radar_trace_simple(orig_profile.loc[cluster_name], cols_orig, color, cluster_name, True), row=1, col=1)
    if cluster_name in mini_profile.index:
        fig_radar_combined.add_trace(make_radar_trace_simple(mini_profile.loc[cluster_name], cols_mini, color, cluster_name, False), row=1, col=2)

    if cluster_name in orig_profile_scaled.index:
        fig_radar_combined.add_trace(make_radar_trace_scaled(orig_profile_scaled.loc[cluster_name], orig_profile.loc[cluster_name], cols_orig, color, cluster_name, False), row=2, col=1)
    if cluster_name in mini_profile_scaled.index:
        fig_radar_combined.add_trace(make_radar_trace_scaled(mini_profile_scaled.loc[cluster_name], mini_profile.loc[cluster_name], cols_mini, color, cluster_name, False), row=2, col=2)

polar_config_abs = dict(radialaxis=dict(visible=True, gridcolor='#555'), angularaxis=dict(gridcolor='#555'))
polar_config_norm = dict(radialaxis=dict(visible=True, range=[0, 1.1], gridcolor='#555', tickvals=[0, 0.5, 1]), angularaxis=dict(gridcolor='#555'))

fig_radar_combined.update_layout(
    title_text="📡 Комплексний математичний портрет кластерів",
    title_x=0.5, template=PLOT_TEMPLATE, width=1500, height=1000,
    polar1=polar_config_abs, polar2=polar_config_abs,
    polar3=polar_config_norm, polar4=polar_config_norm,
    legend=dict(orientation="h", yanchor="bottom", y=-0.05, xanchor="center", x=0.5)
)

print("Красивий Вивід - Пелюсткові діаграми:")
fig_radar_combined.show()

print(f"Технічний Вивід - Метрики стабільності:")
print(f"   🔹 Adjusted Rand Index (ARI): {ari_score:.3f} (1.0 = ідеальний збіг, 0.0 = випадковість)")
print(f"   🔹 Кількість країн, що змінили кластер: {len(changed_df)} ({percent_changed:.2f}%)\n")

if not changed_df.empty:
    changed_df['Зміна'] = changed_df['Cluster_Name'] + " ➔ " + changed_df['Cluster_Mini_Name']
    display_cols = [COUNTRY_COL, 'Зміна', TARGET_METRIC, f1, f2]
    
    print("🌍 Країни, яких ШІ 'перекинув' в інший кластер через нестачу даних:")
    display(changed_df[display_cols].style.set_properties(**TABLE_PROPS).format(precision=3))
else:
    print("🌍 Дивовижно! Усі країни залишилися у своїх кластерах, незважаючи на видалення ознак.")

🔬 Дослідження впливу набору ознак на результати кластеризації...

   ⚙️ Для експерименту зі зниженою розмірністю автоматично обрано: ['Economy..GDP.per.Capita.', 'Family']

Красивий Вивід - Порівняльний графік:


📡 Побудова комплексних аналітичних профілів кластерів...

Красивий Вивід - Пелюсткові діаграми:


Технічний Вивід - Метрики стабільності:
   🔹 Adjusted Rand Index (ARI): 0.229 (1.0 = ідеальний збіг, 0.0 = випадковість)
   🔹 Кількість країн, що змінили кластер: 62 (40.00%)

🌍 Країни, яких ШІ 'перекинув' в інший кластер через нестачу даних:


,Country,Зміна,Happiness.Score,Economy..GDP.per.Capita.,Family
10,Israel,Середній рівень ➔ Високий рівень,7.213,1.375,1.376
11,Costa Rica,Середній рівень ➔ Високий рівень,7.079,1.110,1.416
20,United Arab Emirates,Високий рівень ➔ Середній рівень,6.648,1.626,1.266
21,Brazil,Середній рівень ➔ Високий рівень,6.635,1.107,1.431
22,Czech Republic,Середній рівень ➔ Високий рівень,6.609,1.353,1.434
23,Argentina,Середній рівень ➔ Високий рівень,6.599,1.185,1.440
25,Singapore,Високий рівень ➔ Середній рівень,6.572,1.692,1.354
27,Uruguay,Середній рівень ➔ Високий рівень,6.454,1.218,1.412
29,Panama,Середній рівень ➔ Високий рівень,6.452,1.234,1.373
30,France,Середній рівень ➔ Високий рівень,6.442,1.431,1.388


**13.5.\*\* Класифікація щастя:**

In [20]:
# ======================================================================================+ #
#                🧞‍♂️ Незалежний блок - для тестування збереженої моделі ШІ                 #
#         🧠 Перед виконанням цього блоку обовʼязково зупустіть блок імпортів!!!          #
#   🫵 Локальні константи, вони візьмуться, якщо не будуть задані Глобальні констати!!!   #
# ======================================================================================= #

# 1. Локальні резервні значення (Fallback & Config)
INF_FILE_FOR_TEST		   = "_clustered_result" # Файл для тесту: залишити пустим "" - візьме оригінальний файл | Для тесту на кластеризованому файлі - "_clustered_result"
INF_CONFIDENCE_THRESHOLD   = 60.0         # Поріг сумніву: якщо ймовірність кластера < 60%
INF_NUM_TEST_SAMPLES       = 10           # Кількість країн для масового тестування
INF_RANDOM_SELECTION       = True         # True - випадкові країни, False - по порядку
INF_RANDOM_SEED            = 42           # None - завжди різні країни, 42 - зафіксовані "випадкові"

# Власна вигадана країна (Atlantis) - середні значення для WHR (ВВП, Соціум, Здоров'я, Свобода, Корупція)
INF_CUSTOM_COUNTRY         = [10.5, 0.9, 70.0, 0.8, 0.1]
INF_MODEL_DIR              = globals().get('MODEL_DIR', "GMM_Models")
INF_DATA_DIR               = globals().get('DATA_DIR', "WorldHappinessDataSet")
INF_LOG_PATH               = "whr_predictions_log.csv"
INF_COUNTRY_COL            = globals().get('COUNTRY_COL', 'Country name')

# Визначення року для тестування (Пріоритет глобальним константам)
if 'TARGET_YEAR' in globals():
    INF_TARGET_YEAR = str(globals()['TARGET_YEAR'])
    print(f"✅ Підхоплено ГЛОБАЛЬНИЙ рік для тесту: {INF_TARGET_YEAR}")
else:
    INF_TARGET_YEAR = input("📅 Введіть РІК ДАНИХ для тестування (напр. 2017, 2024): ").strip()

INF_FILE_CSV = os.path.join(INF_DATA_DIR, f"{INF_TARGET_YEAR}{INF_FILE_FOR_TEST}.csv")

# 2. Шляхи (Гібридна логіка: Пріоритет Глобальним -> Пошук у папці -> Дефолт)
if 'MODEL_PATH' in globals():
    INF_FILE_MODEL = globals()['MODEL_PATH']
    print(f"🌍 Підхоплено ГЛОБАЛЬНИЙ шлях до моделі з поточної сесії: {INF_FILE_MODEL}")
else:
    print(f"⚠️ Глобальних констант немає (незалежний запуск). Шукаємо збережені моделі в '{INF_MODEL_DIR}'...")
    available_models = glob.glob(os.path.join(INF_MODEL_DIR, "*.pkl")) if os.path.exists(INF_MODEL_DIR) else []

    if available_models:
        print("📂 Знайдені збережені капсули ШІ:")
        for i, m_path in enumerate(available_models):
            print(f"   [{i}] {os.path.basename(m_path)}")

        m_choice = input(f"🔢 Обери номер моделі (0-{len(available_models)-1}) або натисни Enter для [0]: ").strip()
        m_idx = int(m_choice) if m_choice.isdigit() and int(m_choice) < len(available_models) else 0
        INF_FILE_MODEL = available_models[m_idx]
        print(f"   🎯 Обрано: {os.path.basename(INF_FILE_MODEL)} 🔚")
    else:
        INF_FILE_MODEL = os.path.join(INF_MODEL_DIR, "gmm_whr_brain.pkl")
        print(f"   ❌ Моделей не знайдено. Встановлено базовий шлях пошуку: {INF_FILE_MODEL}")
# ==================================================================================================================

print("🌍 Запуск незалежної системи розпізнавання Кластерів (Inference)...\n")

ram_available = 'gmm_brain' in globals() and 'fitted_scaler' in globals()
disk_available = os.path.exists(INF_FILE_MODEL)

if ram_available and disk_available:
    choice = input("🔀 Модель знайдено в ОЗП і на Диску.\nЗвідки завантажити? (О - ОЗП / Д - Диск) [За замовчуванням: О]: ").strip().lower()
    use_disk = choice in ['д', 'd', 'диск', 'disk']
elif disk_available:
    print("☢️ Модель в ОЗП відсутня. Використовуємо збережену з Диска...")
    use_disk = True
elif ram_available:
    print("☣️ Модель на Диску відсутня. Використовуємо поточну з ОЗП...")
    use_disk = False
else:
    raise SystemExit(f"❌ Критична помилка: Модель не знайдено ні в пам'яті, ні на диску ({INF_FILE_MODEL}).")


# 🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠 🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠
# 🧠    Логіка зчитування памʼяті та капсули    🧠
# 🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠 🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

if use_disk:
    try:
        model_bundle = joblib.load(INF_FILE_MODEL)
        inf_brain        = model_bundle.get('brain')
        inf_scaler       = model_bundle.get('scaler')
        inf_features     = model_bundle.get('features', [])
        inf_labels       = model_bundle.get('cluster_labels', {0: "Низький", 1: "Середній", 2: "Високий"})
        inf_mapping      = model_bundle.get('cluster_mapping', {0:0, 1:1, 2:2})
        inf_colors       = model_bundle.get('ui_colors', ["#ef553b", "#feca57", "#00cc96"])
        inf_algo         = model_bundle.get('algorithm', "GMM")
        inf_year         = model_bundle.get('target_year', "Невідомо")

        file_time = os.path.getmtime(INF_FILE_MODEL)
        model_date = pd.to_datetime(file_time, unit='s').strftime('%d.%m.%Y %H:%M:%S')
        source_msg = "Диск (.pkl капсула)"

        if not os.path.exists(INF_FILE_CSV):
            raise SystemExit(f"❌ Критична помилка: Файл даних '{INF_FILE_CSV}' не знайдено! Перевірте введений рік.")

        inf_df = pd.read_csv(INF_FILE_CSV)

        possible_country_names = ['Country', 'Country name', 'Country or region']
        found_country_col = next((c for c in possible_country_names if c in inf_df.columns), None)
        
        if found_country_col:
            INF_COUNTRY_COL = found_country_col
            inf_df[INF_COUNTRY_COL] = inf_df[INF_COUNTRY_COL].astype(str).str.replace('*', '', regex=False).str.strip()
        else:
            raise ValueError(f"Колонку з країною не знайдено! Доступні колонки у файлі: {list(inf_df.columns)}")

    except Exception as e:
        raise SystemExit(f"❌ Помилка завантаження з диска: {e}")
else:
    inf_brain        = globals()['gmm_brain']
    inf_scaler       = globals()['fitted_scaler']
    inf_features     = globals()['FEATURES_FULL']
    inf_labels       = globals()['CLUSTER_LABELS']
    inf_mapping      = globals().get('cluster_mapping', {0:0, 1:1, 2:2})
    inf_colors       = globals()['COLOR_PALETTE_FULL']
    inf_algo         = "Gaussian Mixture Model (GMM)"
    inf_year         = globals().get('TARGET_YEAR', "Невідомо")
    inf_df           = globals().get('df')

    if inf_df is None or len(inf_df) == 0:
        raise SystemExit("❌ Критична помилка: В оперативній пам'яті відсутній датафрейм (df) для тестування!")

    saved_time = globals().get('_LAST_SAVED_TIMESTAMP_HUMAN')
    model_date = saved_time + " (Збережено)" if saved_time else pd.Timestamp.now().strftime('%d.%m.%Y %H:%M:%S') + " (Не збережена)"
    source_msg = "Оперативна пам'ять (поточна сесія)"

if not inf_brain or not inf_scaler:
    raise SystemExit("\n❌ Неможливо відтворити математику. Відсутній 'Мозок' або Скейлер.")

if len(inf_colors) < inf_brain.n_components:
    inf_colors = [inf_colors[i % len(inf_colors)] for i in range(inf_brain.n_components)]

local_translations = globals().get('FEATURE_TRANSLATIONS', {
    # Економіка
    "Economy (GDP per Capita)": "ВВП на душу населення", "Economy..GDP.per.Capita.": "ВВП на душу населення", "GDP per capita": "ВВП на душу населення", "Logged GDP per capita": "ВВП на душу населення", "Explained by: Log GDP per capita": "ВВП на душу населення", "Explained by: GDP per capita": "ВВП на душу населення",
    # Соціум
	"Family": "Соціальна підтримка", "Social support": "Соціальна підтримка", "Explained by: Social support": "Соціальна підтримка",
	# Здоров'я
    "Health (Life Expectancy)": "Тривалість здорового життя", "Health..Life.Expectancy.": "Тривалість здорового життя", "Healthy life expectancy": "Тривалість здорового життя", "Explained by: Healthy life expectancy": "Тривалість здорового життя",
    # Свобода
    "Freedom": "Свобода вибору", "Freedom to make life choices": "Свобода вибору", "Explained by: Freedom to make life choices": "Свобода вибору",
    # Корупція
    "Trust (Government Corruption)": "Сприйняття корупції", "Trust..Government.Corruption.": "Сприйняття корупції", "Perceptions of corruption": "Сприйняття корупції", "Explained by: Perceptions of corruption": "Сприйняття корупції",
    # Щедрість
    "Generosity": "Щедрість", "Explained by: Generosity": "Щедрість"
})

model_semantics = {local_translations.get(feat, feat): feat for feat in inf_features}
rename_map = {}
for col in inf_df.columns:
    semantic_meaning = local_translations.get(col, col)
    if semantic_meaning in model_semantics and col != model_semantics[semantic_meaning]:
        rename_map[col] = model_semantics[semantic_meaning]

if rename_map:
    inf_df = inf_df.rename(columns=rename_map)
    print(f"\n   🔄 Магія MLOps: Схему даних адаптовано! Перейменовано {len(rename_map)} колонок з {INF_TARGET_YEAR} року під формат {inf_year} року.")

print(f"   🌍 Кількість країн у тестовому наборі даних: {len(inf_df)}\n")


# 📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊 📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
# 📊 Інформаційне досьє на Модель ШІ (Паспорт вмінь) 📊
# 📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊 📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊

print("📚" * 42)
print("   📦 Вміст ШІ - Аналітичне ядро:")
print(f"      🧮 GMM Модель:  ✅ Зчитано ({inf_brain.n_components} кластерів)")
print(f"      📏 Препроцесор: ✅ Зчитано Скейлер")
print(f"      📐 Простір:     {len(inf_features)} ознак")
print("   🧠 Базова інформація про Модель:")
print(f"      📼 Джерело: {source_msg}")
print(f"      🏗️ Архітектура: {inf_algo}")
print(f"      🗓️ Збірка: {model_date}")
print(f"      📆 Рік моделі (на чому вчилась): {inf_year}")
print(f"      🏷️ Категорії: {' | '.join(inf_labels.values())}")
print(f"      🧪 Рік тестування (поточні дані): {INF_TARGET_YEAR}")
print(f"      🌍 Кількість країн для аналізу: {len(inf_df) if inf_df is not None else 0}") # <--- ДОДАНО ОСЬ ТУТ!
print("📚" * 42 + "\n")

user_proceed = input("🚀 Продовжити сканування з цією моделлю? (Так/Ні) [За замовчуванням: Так]: ").strip().lower()
if user_proceed in ['n', 'no', 'н', 'ні']:
    raise SystemExit("🛑 Сканування скасовано.")


# 🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
# 🚨     Математичне Ядро Передбачення (NLP Inference)     🚨
# 🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨

print("   ✅ Ядро ініціалізовано. Починаємо аналіз...\n")

tasks = []
log_entries = []

safe_atlantis_features = (INF_CUSTOM_COUNTRY + [0.0] * 10)[:len(inf_features)]

tasks.append({
    'id': 'MYSTERY_ATLANTIS', 
    'country': 'Atlantis',
    'features': safe_atlantis_features,
    'true_label': 'unknown',
    'model_year': inf_year,
    'data_year': 'Вигаданий'
})

print("   🔍 Формування черги на сканування...")
if inf_df is not None and len(inf_df) > 0:
    sample_size = min(INF_NUM_TEST_SAMPLES, len(inf_df))
    if INF_RANDOM_SELECTION:
        test_df = inf_df.sample(n=sample_size, random_state=INF_RANDOM_SEED)
        print("       🔀 Режим: ВИПАДКОВІ країни з набору.")
    else:
        test_df = inf_df.head(sample_size)
        print("       ⬇️ Режим: ПО ПОРЯДКУ (послідовне читання).")

    missing_cols = [f for f in inf_features if f not in inf_df.columns]
    if missing_cols:
        print(f"       ❌ УВАГА: У наборі даних відсутні потрібні колонки: {missing_cols}")
        print("       ⚠️ Спробуй перейменувати колонки у сирому наборі даних, щоб вони збігалися з моделлю.")
    else:
        test_df_clean = test_df.copy()

        for col in inf_features:
            test_df_clean[col] = pd.to_numeric(test_df_clean[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

        imputer = SimpleImputer(strategy='median')
        test_df_clean[inf_features] = imputer.fit_transform(test_df_clean[inf_features])

        for _, row in test_df_clean.iterrows():
            c_name = row[INF_COUNTRY_COL] if INF_COUNTRY_COL in row.index else "Невідома країна"
            true_cls = row['Cluster_Name'] if 'Cluster_Name' in row.index else 'unknown'
            f_vec = [row[f] for f in inf_features]
            tasks.append({
                'id': f'SCAN_{str(c_name)[:3].upper()}', 
                'country': c_name, 
                'features': f_vec, 
                'true_label': true_cls, 
                'model_year': inf_year,
                'data_year': str(INF_TARGET_YEAR)
            })
else:
    print("       ⚠️ Джерела даних відсутні (набір даних не знайдено). Працюємо лише з вигаданою країною.")


# 🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀
# 🎀     HTML: Текстові віджети (Prediction Cards)       🎀
# 🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀🎀

time_start = time.time()

display(HTML("""
<style>
.tooltip { position: relative; display: inline-block; border-bottom: 1px dotted #00ffcc; cursor: help; }
.tooltip .tooltiptext { visibility: hidden; width: 220px; background-color: #333; color: #fff; text-align: center; border-radius: 6px; padding: 5px; position: absolute; z-index: 1; bottom: 125%; left: 50%; margin-left: -110px; opacity: 0; transition: opacity 0.3s; font-size: 11px; font-weight: normal; font-family: sans-serif; box-shadow: 0px 0px 5px rgba(0,0,0,0.5); }
.tooltip .tooltiptext::after { content: ""; position: absolute; top: 100%; left: 50%; margin-left: -5px; border-width: 5px; border-style: solid; border-color: #333 transparent transparent transparent; }
.tooltip:hover .tooltiptext { visibility: visible; opacity: 1; }
</style>
"""))

for task in tasks:
    f_vec = task['features']
    country = task['country']
    true_label_name = task['true_label']

    expected_cols = getattr(inf_scaler, 'feature_names_in_', inf_features)

    if len(f_vec) != len(expected_cols):
        print(f"   ❌ БЛОКУВАННЯ: '{country}' пропущено! Модель вимагає {len(expected_cols)} вимірів, а передано {len(f_vec)}.")
        print(f"      👉 РІШЕННЯ: Якщо ви змінили кількість ознак в ACTIVE_DIMENSIONS, перезапустіть блоки 'Масштабування' та 'Навчання'!")
        continue

    input_df = pd.DataFrame([f_vec], columns=expected_cols)
    f_scaled = inf_scaler.transform(input_df)
    f_scaled_df = pd.DataFrame(f_scaled, columns=expected_cols)

    raw_class_idx = inf_brain.predict(f_scaled_df)[0]
    raw_probs = inf_brain.predict_proba(f_scaled_df)[0]

    pred_class_idx = inf_mapping.get(raw_class_idx, raw_class_idx)

    probs = np.zeros_like(raw_probs)
    for raw_id, new_id in inf_mapping.items():
        if raw_id < len(raw_probs):
            probs[new_id] = raw_probs[raw_id]

    pred_name_ui = inf_labels.get(pred_class_idx, f"Кластер {pred_class_idx+1}")
    pred_color = inf_colors[pred_class_idx] if pred_class_idx < len(inf_colors) else "#ffffff"

    is_mystery = (true_label_name == 'unknown')
    max_conf = np.max(probs) * 100
    is_anomaly = max_conf < INF_CONFIDENCE_THRESHOLD
    is_correct = (pred_name_ui == true_label_name) if not is_mystery else None

    if is_mystery:
        status_icon = "👽 АНАЛІЗ"
        status_color = "#00bfff"
        log_status = "N/A"
    elif is_anomaly:
        status_icon = "🤷‍♂️ СУМНІВ"
        status_color = "#ff8c00"
        log_status = "Сумнів"
    else:
        status_icon = "😎 УСПІХ" if is_correct else "❌ ХИБНО"
        status_color = "#55ff55" if is_correct else "#ff4444"
        log_status = "Вірно" if is_correct else "Хибно"

    bars_html = ""
    prob_labels_html = ""
    for i in range(inf_brain.n_components):
        p_val = probs[i]
        c_color = inf_colors[i] if i < len(inf_colors) else "#888"
        short_name = inf_labels.get(i, f"Cls{i}").split(' ')[0]

        bars_html += f'<div style="width: {p_val*100}%; background: {c_color};"></div>'
        prob_labels_html += f'<span>{short_name}: {p_val*100:.2f}%</span>'

    feat_html = ""
    for i, val in enumerate(f_vec):
        raw_name = inf_features[i]
        ua_name = local_translations.get(raw_name, raw_name)
        ua_abbr = "".join([w[0].upper() for w in ua_name.split() if w.isalpha()])[:3]
        if not ua_abbr: 
            ua_abbr = f"ОЗ{i+1}"

        feat_html += f'<div><span class="tooltip">{ua_abbr}<span class="tooltiptext">{ua_name}</span></span>: {val:.2f}</div>'

    html_card = f"""
    <hr style="height: 2px; border: none; background: linear-gradient(90deg, transparent, #007bff, transparent); opacity: 0.8;">
    <div style="background-color: #111; padding: 15px 20px; border-radius: 12px; border: 1px solid #333; display: flex; align-items: stretch; justify-content: space-between; font-family: 'Consolas', monospace; margin-top: 10px;">
        <div style="flex: 2; padding-right: 20px;">
            <div style="color: #888; font-size: 11px; text-transform: uppercase;">
                SCAN ID: {task['id']} | 🧠 МОДЕЛЬ: {task['model_year']} | 📅 ДАНІ ТЕСТУ: {task['data_year']}
            </div>
            <div style="color: #fff; font-size: 16px; font-weight: bold; margin-top: 5px;">🌍 {country}</div>
            <div style="display: flex; gap: 15px; margin-top: 10px; color: #00ffcc; font-size: 13px;">
                {feat_html}
            </div>
        </div>
        <div style="flex: 2; margin: 0 15px; border-left: 1px dotted #444; padding-left: 15px;">
            <div style="color: #888; font-size: 10px; margin-bottom: 5px;">[ ЙМОВІРНІСТЬ GMM (SOFTMAX) ]</div>
            <div style="background: #000; border-radius: 2px; height: 10px; display: flex; overflow: hidden; border: 1px solid #444;">
                {bars_html}
            </div>
            <div style="font-size: 10px; margin-top: 5px; display: flex; justify-content: space-between; color: #aaa;">
                {prob_labels_html}
            </div>
        </div>
        <div style="flex: 1.5; text-align: right; border-right: 4px solid {status_color}; padding-right: 15px;">
            <div style="color: {status_color}; font-size: 16px; font-weight: bold;">{status_icon}</div>
            <div style="color: {pred_color}; font-size: 16px; font-weight: bold; margin-top: 5px; text-shadow: 0 0 8px {pred_color}50;">ШІ: {pred_name_ui}</div>
            <div style="color: #888; font-size: 12px; margin-top: 3px;">Впевненість: {max_conf:.2f}%</div>
        </div>
    </div>
    """
    display(HTML(html_card))

    if is_mystery:
        print("   👽 Користувацька країна (Свій зразок без мітки)")
        print(f"      🧚‍♀️ ШІ бачить тут: {pred_name_ui} ({max_conf:.2f}%)")
    elif is_anomaly:
        print(f"   🤷‍♂️ ШІ розгубився (Впевненість {max_conf:.2f}% — нижче порогу {INF_CONFIDENCE_THRESHOLD}%)")
        print(f"      🗺️ Справжній кластер: {true_label_name}")
        print(f"      🐣 Найближча здогадка була: {pred_name_ui}")
    elif not is_mystery:
        icon = "✅" if is_correct else "❌"
        status = "🎉 Вгадав!" if is_correct else "🫠 Не вгадав!"
        print(f"   {icon} Знайома країна: {country} ({true_label_name})")
        print(f"      🧞‍♂️ Вердикт ШІ: {pred_name_ui} ({max_conf:.2f}%)\n         {status}")
    else:
        print(f"   🤔 Країна невідомої категорії: {country}")
        print(f"      🧜‍♀️ ШІ припускає: {pred_name_ui} ({max_conf:.2f}%)")

    prob_parts = [f"{str(inf_labels.get(i, i))[:3]}:{probs[i]*100:.2f}%" for i in range(len(probs))]
    prob_str = " | ".join(prob_parts)

    log_entries.append({
        "Дата": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Країна": country,
        "Справжній Кластер": true_label_name,
        "Рік Моделі": task['model_year'],
        "Рік Даних": task['data_year'],
        "Прогноз ШІ": pred_name_ui,
        "Впевненість (%)": round(max_conf, 1),
        "Статус": log_status,
        "Розподіл": prob_str
    })

total_time = time.time() - time_start
known_preds = [item for item in log_entries if item['Справжній Кластер'] != "unknown"]

total_known = len(known_preds)
correct_count = sum(1 for item in known_preds if item['Статус'] == "Вірно")
acc_ours = (correct_count / total_known * 100) if total_known > 0 else 0

print("\n" + "⭐" * 50)
print("📊 Підсумок компетентності Системи:")
print(f"   ⏱️ Швидкість Інференсу: Загалом {total_time:.3f} сек")
if total_known > 0:
    print(f"   🎯 Точність збігу міток: {acc_ours:.2f}% ({correct_count} з {total_known} відомих країн)")
    print(f"   💡 Примітка: 100% збіг очікується, якщо ШІ аналізує ті ж дані, на яких навчався.")
print("⭐" * 50)

if log_entries:
    df_new = pd.DataFrame(log_entries)
    if os.path.exists(INF_LOG_PATH):
        try:
            df_old = pd.read_csv(INF_LOG_PATH)
            df_combined = pd.concat([df_old, df_new], ignore_index=True)
            df_combined.to_csv(INF_LOG_PATH, index=False, encoding='utf-8-sig')
        except:
            df_new.to_csv(INF_LOG_PATH, index=False, encoding='utf-8-sig')
    else:
        df_new.to_csv(INF_LOG_PATH, index=False, encoding='utf-8-sig')

    print(f"\n💾 Детальний журнал ідентифікації успішно оновлено: {INF_LOG_PATH}\n")

✅ Підхоплено ГЛОБАЛЬНИЙ рік для тесту: 2017
🌍 Підхоплено ГЛОБАЛЬНИЙ шлях до моделі з поточної сесії: GMM_Models/gmm_2017_full_kmeans_model.pkl
🌍 Запуск незалежної системи розпізнавання Кластерів (Inference)...

   🌍 Кількість країн у тестовому наборі даних: 155

📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚
   📦 Вміст ШІ - Аналітичне ядро:
      🧮 GMM Модель:  ✅ Зчитано (3 кластерів)
      📏 Препроцесор: ✅ Зчитано Скейлер
      📐 Простір:     6 ознак
   🧠 Базова інформація про Модель:
      📼 Джерело: Оперативна пам'ять (поточна сесія)
      🏗️ Архітектура: Gaussian Mixture Model (GMM)
      🗓️ Збірка: 01.04.2026 01:57:29 (Збережено)
      📆 Рік моделі (на чому вчилась): 2017
      🏷️ Категорії: Низький рівень | Середній рівень | Високий рівень
      🧪 Рік тестування (поточні дані): 2017
      🌍 Кількість країн для аналізу: 155
📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚📚

   ✅ Ядро ініціалізовано. Починаємо аналіз...

   🔍 Формування черги на сканування...
       🔀 Режим: ВИПАДКОВІ країни 

   👽 Користувацька країна (Свій зразок без мітки)
      🧚‍♀️ ШІ бачить тут: Низький рівень (100.00%)


   ✅ Знайома країна: Venezuela (Середній рівень)
      🧞‍♂️ Вердикт ШІ: Середній рівень (99.76%)
         🎉 Вгадав!


   ✅ Знайома країна: Benin (Низький рівень)
      🧞‍♂️ Вердикт ШІ: Низький рівень (100.00%)
         🎉 Вгадав!


   ✅ Знайома країна: Thailand (Середній рівень)
      🧞‍♂️ Вердикт ШІ: Середній рівень (99.76%)
         🎉 Вгадав!


...

   ✅ Знайома країна: North Cyprus (Середній рівень)
      🧞‍♂️ Вердикт ШІ: Середній рівень (98.91%)
         🎉 Вгадав!


   ✅ Знайома країна: Vietnam (Середній рівень)
      🧞‍♂️ Вердикт ШІ: Середній рівень (99.59%)
         🎉 Вгадав!


   ✅ Знайома країна: Liberia (Низький рівень)
      🧞‍♂️ Вердикт ШІ: Низький рівень (98.82%)
         🎉 Вгадав!


   ✅ Знайома країна: Burundi (Низький рівень)
      🧞‍♂️ Вердикт ШІ: Низький рівень (100.00%)
         🎉 Вгадав!


   ✅ Знайома країна: Turkey (Середній рівень)
      🧞‍♂️ Вердикт ШІ: Середній рівень (99.80%)
         🎉 Вгадав!

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
📊 Підсумок компетентності Системи:
   ⏱️ Швидкість Інференсу: Загалом 0.214 сек
   🎯 Точність збігу міток: 100.00% (10 з 10 відомих країн)
   💡 Примітка: 100% збіг очікується, якщо ШІ аналізує ті ж дані, на яких навчався.
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐

💾 Детальний журнал ідентифікації успішно оновлено: whr_predictions_log.csv



**14. Висновок:**

# Інференс та Аналітичні Висновки: Ймовірнісна кластеризація та MLOps-архітектура глобального рівня життя (WHR)

У цій роботі розроблено повноцінний аналітичний MLOps-пайплайн для неконтрольованого навчання (Unsupervised Learning). Ми застосували імовірнісний підхід на базі алгоритму очікування-максимізації (EM) для налаштування параметрів Моделі Сумішей Гаусса (GMM), повністю відтворивши архітектуру від аналізу сирих даних (EDA) до готового Inference-модуля.

### 1. Топологія простору та Ймовірнісне мислення (Soft Clustering)

Аналіз (Крок 5) показав, що реальні соціально-економічні дані далекі від ідеального Гаусівського розподілу: ознаки `Economy` та `Health` мають лівосторонню асиметрію, а `Generosity` та `Trust` — експоненційний характер спадання. Проте, GMM успішно апроксимувала ці складні форми, використовуючи суміш нормальних розподілів.

На відміну від K-Means, щільність ймовірності для кожної країни $\mathbf{x}$ описувалася функцією:

$$\mathcal{N}(\mathbf{x} | \boldsymbol{\mu}_c, \boldsymbol{\Sigma}_c) = \frac{1}{\sqrt{(2\pi)^D |\boldsymbol{\Sigma}_c|}} \exp\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu}_c)^T \boldsymbol{\Sigma}_c^{-1} (\mathbf{x} - \boldsymbol{\mu}_c)\right)$$

Завдяки оптимізації через EM-алгоритм, ми отримали Softmax-масив імовірностей приналежності країни до кожного рівня життя (E-крок):

$$\gamma_{ic} = \frac{\pi_c \mathcal{N}(\mathbf{x}_i | \boldsymbol{\mu}_c, \boldsymbol{\Sigma}_c)}{\sum_{j=1}^K \pi_j \mathcal{N}(\mathbf{x}_i | \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

---

### 2. Мультиколінеарність та сенс ізотропії (Крок 7 та 10)

Кореляційна матриця виявила сильний лінійний зв'язок між векторами `Economy`, `Health` та `Family` ($r \approx 0.75 - 0.81$). Оскільки просторові вектори майже співнаправлені, використання повної матриці коваріацій (`covariance_type='full'`) дозволило кластерам набувати форми витягнутих багатовимірних еліпсоїдів.

**Математичний сенс стандартизації:** Якби ми залишили оригінальні дані (де ВВП $\in [0, 1.8]$, а Довіра $\in [0, 0.4]$), матриця коваріацій $\boldsymbol{\Sigma}$ була б погано обумовленою, і дисперсія макроекономіки повністю б "заглушила" вплив довіри до уряду. Застосування `StandardScaler` відцентрувало дані ($\boldsymbol{\mu} \approx 0$) та нормалізувало дисперсію ($\sigma \approx 1$), створивши ізотропний простір для коректної оптимізації.

---

### 3. Вплив розмірності простору та Математичні портрети (Кроки 13-14)

- **Ground Truth (Повний набір - 5 ознак):** Кластеризація на три ідейні зони географічно та економічно майже ідеально відтворила оригінальну теплову мапу `Happiness.Score`. **Архітектурний успіх:** Модель виявила ці геополітичні закономірності виключно через внутрішню коваріацію ознак, **не маючи жодного доступу до цільової змінної**. 
- **Втрата роздільної здатності (Міні-набір - 2 ознаки):** Зменшення розмірності до базових ВВП та Здоров'я зруйнувало ландшафт кластерів. Країни з високим індексом соціальних зв'язків, але середнім ВВП (як у Південній Америці), були помилково віднесені до нижчих рівнів. 
- **Радарні діаграми:** Застосування `MinMaxScaler` до агрегованих профілів ($X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$) дозволило усунути масштабні викривлення. Ми наочно побачили, що "Високий рівень життя" формується не лише економікою, але й завдяки майже 100% заповненню "пелюсток" соціальної довіри та свободи.

**Стиснення простору (PCA) та Латентні зв'язки:**

Для візуальної інтерпретації 5-вимірного простору ми застосували Метод головних компонент (PCA). Ортогональне перетворення дозволило спроєктувати Гаусівські еліпсоїди на 2D-площину з мінімальною втратою інформативності (збережено понад 70% дисперсії в перших двох компонентах). Це довело, що перша головна компонента (PC1) фактично виступає прихованим (латентним) фактором "Загального добробуту", вздовж якого моделі GMM було найлегше провести розділяючі гіперплощини між кластерами.

---

### 4. Інженерна зрілість та Шляхи масштабування для Production

Ми успішно вирішили проблему розсинхронізації стану моделі під час інференсу. Збереження матриць коваріації $\boldsymbol{\Sigma}$, векторів середніх $\boldsymbol{\mu}$, параметрів нормалізації (`fitted_scaler`) та метаданих в єдину `.pkl` капсулу гарантує ідеальну відтворюваність. Розроблений пайплайн здатен безперебійно приймати нові сирі дані, застосовувати трансформацію та повертати Softmax-ймовірності.

**MLOps / Architect Perspective:**

1. **Anomaly Detection (Out-of-Distribution):** GMM повертає логарифм правдоподібності `score_samples(X)`. У Production це дозволить системі виявляти аномалії — країни з унікальними економічними моделями, що лежать за межами встановлених Гаусівських розподілів (поріг `CONFIDENCE_THRESHOLD`).
2. **Online EM-Algorithm:** Для систем, де дані надходять потоком безперервно (Data Streaming), стандартний `GaussianMixture` (що потребує всього набору даних в RAM) доцільно замінити на стохастичні (online) версії EM-алгоритму або Байєсівські суміші (Bayesian GMM), що автоматично визначають оптимальну кількість кластерів $K$ через процеси Діріхле.

---

### 5. Архітектура розробленого MLOps Пайплайну (Data Flow Diagram)

```mermaid
graph TD
    classDef data fill:#2b2b2b,stroke:#00c3ff,stroke-width:2px,color:#fff;
    classDef process fill:#1a1a1a,stroke:#ccff00,stroke-width:2px,color:#fff;
    classDef model fill:#1a1a1a,stroke:#ff007f,stroke-width:2px,color:#fff;
    classDef output fill:#222,stroke:#00ffcc,stroke-width:2px,color:#fff;
    classDef inference fill:#1a1a1a,stroke:#ff8c00,stroke-width:2px,color:#fff;

    %% --- TRAINING PIPELINE ---
    A[(Сирі дані WHR)]:::data --> B{Обробка NaN}:::process
    B -->|SimpleImputer| C(Масштабування ознак):::process
    C -->|StandardScaler| D[Ізотропний X_train]:::data

    D --> E((Gaussian Mixture Model)):::model
    E -->|Expectation-Maximization| F{Матриці Коваріації Σ}:::process

    F --> G[Кластер 0: Низький]:::output
    F --> H[Кластер 1: Середній]:::output
    F --> I[Кластер 2: Високий]:::output

    G --> J(Аналітика: Radar Charts & PCA):::process
    H --> J
    I --> J

    J --> K[Нормалізовані Профілі 0-1]:::output
    G --> L[Choropleth: Карта Світу]:::output
    H --> L
    I --> L

    E --> M[(joblib: Серіалізація 'Мозку')]:::data
    C --> M

    %% --- INFERENCE PIPELINE ---
    N[Нові дані / Атлантида]:::data -.-> O[Завантаження Капсули]:::inference
    M -.-> O
    O -.-> P{Синхронізація Схем}:::inference
    P -.->|Softmax| Q[Прогноз: Ймовірність кластерів %]:::output
```